# Modelo térmico reduzido do Coil Temps

Este notebook implementa, em scikit-fem, o modelo de elementos finitos reduzido da área Coil Temps. Ele é o motor das features físicas do Nível 3: nos cards seguintes, o modelo é calibrado sobre os ensaios de treino e suas temperaturas passam a alimentar o Random Forest de produção. O modelo não substitui o aprendizado de máquina, não é uma simulação de CFD e não usa malha tridimensional.

**Finalidade.** O trabalho existe para atacar o risco R06 da Seção 4.1.5 da documentação: uma subestimação de temperatura ainda dentro da margem de 10 por cento contratada consome a folga de segurança da regra de projeto e aprova uma arquitetura que ultrapassa o limite. O risco se concentra nos componentes de limite 150 °C e 220 °C, que são os desta área. A linha de base medida no notebook `colabs_S3/linha_base_kpi.ipynb` localiza o problema: o Coil Temps tem 65,41 por cento das predições dentro da margem e 19 falsos aprovados, dos quais 15 nas placas de alumínio. Os critérios de sucesso dos cards seguintes são medidos contra essas grandezas, e não contra o erro médio.

**Escopo físico.** Condução de calor em regime permanente, em duas dimensões, com geração volumétrica na resistência e contorno convectivo de Robin. A radiação fica fora desta entrega, conforme registrado na Seção 6.

**Dois objetos térmicos.** O domínio é tratado como dois objetos distintos: as quatro resistências (Coil FL, FR, RL e RR, limite ativo de 220 °C) e as quatro placas (Aluminium Plate FL, FR, RL e RR, limite ativo de 150 °C). A justificativa está na Seção 6.

**Limitação de validação.** Os ensaios da base provêm de simulação, e não de bancada física. Calibrar e conferir um modelo físico contra eles mede consistência entre simulações, e não realismo. A verificação deste notebook é feita contra solução analítica fechada, que não depende da base.

**Pré-requisitos.** O arquivo `limites_ativos.csv`, gerado pelo notebook `colabs_S3/preparacao_datasets.ipynb`, e o pacote `scikit-fem` instalado no ambiente virtual do repositório. Nenhum arquivo de dados é versionado.

## 1. Ambiente

As versões são impressas para que a execução possa ser reproduzida. Todas as unidades seguem o Sistema Internacional, com temperaturas em graus Celsius; como o problema é linear e não inclui radiação, diferenças de temperatura em Celsius e em kelvin coincidem.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S4/testes_extras/fem_coil_temps"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import os

import numpy as np
import pandas as pd
import scipy
import scipy.sparse as sp
from scipy.sparse.linalg import spsolve

import skfem
from skfem import (MeshTri, Basis, FacetBasis, ElementTriP1,
                   BilinearForm, LinearForm, Functional, asm, solve, condense)
from skfem.helpers import dot, grad

print("numpy", np.__version__, "| scipy", scipy.__version__,
      "| pandas", pd.__version__, "| scikit-fem", skfem.__version__)

## 2. Limites ativos e linha de aprovação

Os limites dos oito alvos são lidos de `limites_ativos.csv`. A linha de aprovação é 0,9 vez o limite, que é a regra de projeto da engenharia descrita no risco R06: a arquitetura só é aprovada quando a temperatura fica 10 por cento abaixo do limite do componente. A função canônica de avaliação usa hoje o limite absoluto; a linha de 0,9 vez o limite é a que os Cards 2 e 3 consomem.

A tabela registra também a qual objeto térmico cada alvo pertence. Nas placas, o nome do termopar não indica o material: a leitura existe também nos ensaios de suporte plástico, e o material vem da coluna `material_coil_support`.

In [ ]:
BASES = [".", "..", os.path.join("..", ".."), "/content",
         os.path.join("..", "..", "..", "colabs_S3"), os.path.join("notebooks", "colabs_S3"),
         os.path.join("..", "..", "..", "colabs_S3", "preparacao_testes"), os.path.join("notebooks", "colabs_S3", "preparacao_testes")]


def resolve(arquivo):
    tentativas = [os.path.join(b, arquivo) for b in BASES]
    for caminho in tentativas:
        if os.path.exists(caminho):
            return caminho
    raise FileNotFoundError(f"{arquivo} nao encontrado. Diretorio atual: {os.path.abspath('.')}. "
                            "Caminhos tentados: " + "; ".join(tentativas))


ALVOS = {
    "Coil FL": "resistencia", "Coil FR": "resistencia",
    "Coil RL": "resistencia", "Coil RR": "resistencia",
    "Aluminium Plate FL": "placa", "Aluminium Plate FR": "placa",
    "Aluminium Plate RL": "placa", "Aluminium Plate RR": "placa",
}
FATOR_APROVACAO = 0.9   # regra de projeto: 10 por cento abaixo do limite (risco R06)

_lim = pd.read_csv(resolve("limites_ativos.csv"))
LIMITES_COIL = (_lim[_lim["tc_name"].isin(ALVOS)]
                .loc[:, ["tc_name", "key_area", "limite_num", "apenas_monitorar"]]
                .rename(columns={"tc_name": "componente", "limite_num": "limite_C"})
                .assign(objeto=lambda t: t["componente"].map(ALVOS),
                        linha_aprovacao_C=lambda t: FATOR_APROVACAO * t["limite_C"])
                .sort_values(["objeto", "componente"])
                .reset_index(drop=True))

assert len(LIMITES_COIL) == 8, f"esperados 8 alvos, encontrados {len(LIMITES_COIL)}"
assert LIMITES_COIL["limite_C"].notna().all() and not LIMITES_COIL["apenas_monitorar"].any()
assert (LIMITES_COIL.loc[LIMITES_COIL.objeto == "resistencia", "limite_C"] == 220).all()
assert (LIMITES_COIL.loc[LIMITES_COIL.objeto == "placa", "limite_C"] == 150).all()

LIMITE = LIMITES_COIL.set_index("componente")["limite_C"].to_dict()
LINHA_APROVACAO = LIMITES_COIL.set_index("componente")["linha_aprovacao_C"].to_dict()
print(LIMITES_COIL.to_string(index=False))

## 3. Propriedades de material e coeficientes de partida

Todos os valores estão em unidades do Sistema Internacional e vêm de BERGMAN et al. (2014), *Fundamentos de Transferência de Calor e de Massa*, 7. ed., obra usualmente citada pelo nome de Incropera. As propriedades de sólidos são as tabeladas a 300 K. O regime é permanente, de modo que apenas a condutividade térmica entra nas equações; massa específica e calor específico só seriam necessários no regime transiente, fora deste card.

| Símbolo | Material ou grandeza | Valor | Unidade | Fonte |
|---|---|---:|---|---|
| `K_AL` | Alumínio puro | 237 | W/(m·K) | Tabela A.1 |
| `K_AL_2024` | Liga de alumínio 2024-T6 | 177 | W/(m·K) | Tabela A.1 |
| `K_NICROMO` | Nicromo, 80% Ni e 20% Cr | 12 | W/(m·K) | Tabela A.1 |
| `K_TEFLON` | Teflon, a 300 K | 0,35 | W/(m·K) | Tabela A.3 |
| `K_TEFLON_400` | Teflon, a 400 K | 0,45 | W/(m·K) | Tabela A.3 |
| `K_BORRACHA` | Borracha vulcanizada macia | 0,13 | W/(m·K) | Tabela A.3 |
| `K_BORRACHA_DURA` | Borracha vulcanizada dura | 0,16 | W/(m·K) | Tabela A.3 |
| `H_AR_LIVRE` | Convecção livre em gases | 2 a 25 | W/(m²·K) | Tabela 1.1 |
| `H_AR_FORCADA` | Convecção forçada em gases | 25 a 250 | W/(m²·K) | Tabela 1.1 |
| `R_CONTATO` | Resistência de contato, interface de alumínio com ar | 2,75 × 10⁻⁴ | m²·K/W | Tabela 3.1 |

Quatro escolhas acompanham a tabela e são declaradas como hipóteses, não como dado da parceira.

- **Placa.** O suporte metálico é modelado com o alumínio puro, e a liga 2024-T6 fica como limite de sensibilidade, pois a liga efetivamente usada não foi informada. O suporte plástico é modelado com o Teflon, e a borracha vulcanizada fica como limite inferior de condutividade. Os dois são aproximações de um polímero não especificado.
- **Resistência.** O material do elemento aquecedor não foi informado. O nicromo é a liga resistiva típica e é a adotada como valor de partida.
- **Convecção.** O valor de partida é 10 W/(m²·K), dentro da faixa de convecção livre. O coeficiente é calibrado no Card 2, e valor calibrado fora das faixas da Tabela 1.1 é sinalizado, não silenciado.
- **Contato.** A condutância entre resistência e placa parte do inverso da resistência de contato da tabela. O valor tabelado é para interface entre superfícies de alumínio; para o suporte plástico, a resistência do próprio polímero domina a do contato, de modo que o mesmo valor é mantido.

A numeração das tabelas deve ser conferida na edição adotada pelo grupo antes do registro da referência na Seção 6 da documentação.

In [ ]:
# Condutividade termica, W/(m.K), a 300 K salvo indicacao (Incropera, Tabelas A.1 e A.3)
K_AL = 237.0            # aluminio puro, Tabela A.1
K_AL_2024 = 177.0       # liga 2024-T6, Tabela A.1
K_NICROMO = 12.0        # nicromo 80Ni-20Cr, Tabela A.1
K_TEFLON = 0.35         # Teflon a 300 K, Tabela A.3
K_TEFLON_400 = 0.45     # Teflon a 400 K, Tabela A.3
K_BORRACHA = 0.13       # borracha vulcanizada macia, Tabela A.3
K_BORRACHA_DURA = 0.16  # borracha vulcanizada dura, Tabela A.3

# Coeficiente de conveccao, W/(m2.K) (Incropera, Tabela 1.1)
H_AR_LIVRE = (2.0, 25.0)
H_AR_FORCADA = (25.0, 250.0)
H_AR_PARTIDA = 10.0     # valor de partida dentro da conveccao livre; calibrado no Card 2

# Contato entre resistencia e placa (Incropera, Tabela 3.1, interface de aluminio com ar)
R_CONTATO = 2.75e-4                # m2.K/W
H_CONTATO = 1.0 / R_CONTATO        # W/(m2.K)

K_RESISTENCIA = K_NICROMO
K_PLACA = {"aluminium": K_AL, "plastic": K_TEFLON}   # chaves de material_coil_support

T_AMBIENTE = 25.0       # C; hipotese de ambiente, nao informada pela parceira


def faixa_h(h):
    """Classifica um coeficiente de conveccao contra as faixas da Tabela 1.1."""
    if H_AR_LIVRE[0] <= h <= H_AR_LIVRE[1]:
        return "conveccao livre"
    if H_AR_FORCADA[0] < h <= H_AR_FORCADA[1]:
        return "conveccao forcada"
    return "FORA DA FAIXA FISICA"


propriedades = pd.DataFrame([
    ("K_AL", K_AL, "W/(m.K)"), ("K_AL_2024", K_AL_2024, "W/(m.K)"),
    ("K_NICROMO", K_NICROMO, "W/(m.K)"), ("K_TEFLON", K_TEFLON, "W/(m.K)"),
    ("K_TEFLON_400", K_TEFLON_400, "W/(m.K)"), ("K_BORRACHA", K_BORRACHA, "W/(m.K)"),
    ("K_BORRACHA_DURA", K_BORRACHA_DURA, "W/(m.K)"),
    ("H_AR_PARTIDA", H_AR_PARTIDA, "W/(m2.K)"), ("H_CONTATO", H_CONTATO, "W/(m2.K)"),
    ("T_AMBIENTE", T_AMBIENTE, "C"),
], columns=["simbolo", "valor", "unidade"])
print(propriedades.to_string(index=False))
print("\nH_AR_PARTIDA:", faixa_h(H_AR_PARTIDA))

## 4. Formulação e formas em scikit-fem

Em cada objeto, a temperatura $T$ satisfaz a equação de condução em regime permanente com geração volumétrica $\dot q$:

$$-\nabla \cdot (k \nabla T) = \dot q \quad \text{em } \Omega$$

com contorno convectivo de Robin

$$-k \frac{\partial T}{\partial n} = h \,(T - T_{ref}) \quad \text{em } \Gamma_R$$

e, apenas no teste de verificação, temperatura prescrita em parte do contorno. Faces sem condição declarada são adiabáticas. Multiplicando por uma função teste $v$ e integrando por partes, obtém-se a forma fraca

$$\int_\Omega k \,\nabla T \cdot \nabla v \, d\Omega + \int_{\Gamma_R} h \, T v \, d\Gamma = \int_\Omega \dot q \, v \, d\Omega + \int_{\Gamma_R} h \, T_{ref} \, v \, d\Gamma$$

O primeiro termo é a forma bilinear de condução; o segundo, a forma bilinear do contorno convectivo; os dois termos da direita são as formas lineares de geração e de convecção. O elemento é o triângulo linear.

A matriz do contorno convectivo é agrupada na diagonal. Com a matriz consistente, e com número de Biot por elemento elevado, caso do contato com a placa plástica, a solução discreta viola o princípio do máximo: a placa, sem fonte própria, chegou a ficar mais quente que a face da resistência que a aquece. O agrupamento restaura o princípio sem alterar a solução dos casos em que a temperatura é constante ao longo da face, como o do teste analítico.

As mesmas funções são usadas no teste de verificação e no modelo dos dois objetos, de modo que o teste confere o código efetivamente usado.

In [ ]:
ELEMENTO = ElementTriP1()


@BilinearForm
def conducao(u, v, w):
    return w.k * dot(grad(u), grad(v))


@LinearForm
def geracao(v, w):
    return w.q * v


@BilinearForm
def conveccao_bilinear(u, v, w):
    return w.h * u * v


@LinearForm
def conveccao_linear(v, w):
    return w.h * w.T_ref * v


@LinearForm
def comprimento(v, w):
    return 1.0 * v


@Functional
def fluxo_convectivo(w):
    return w.h * (w.T - w.T_ref)


@Functional
def integral_T(w):
    return w.T


def montar(mesh, k, q=0.0, conveccoes=()):
    """Matriz e vetor de um objeto.

    conveccoes: lista de (faces, h, T_ref). Com T_ref None, entra apenas o termo
    bilinear, e a referencia e fornecida pelo acoplamento entre objetos.
    """
    basis = Basis(mesh, ELEMENTO)
    A = asm(conducao, basis, k=k)
    b = asm(geracao, basis, q=q)
    for faces, h, T_ref in conveccoes:
        fb = FacetBasis(mesh, ELEMENTO, facets=faces)
        R = asm(conveccao_bilinear, fb, h=h)
        A = A + sp.diags(np.asarray(R.sum(axis=1)).ravel())   # matriz agrupada
        if T_ref is not None:
            b = b + asm(conveccao_linear, fb, h=h, T_ref=T_ref)
    return basis, A, b


def resolver(mesh, k, q=0.0, conveccoes=(), prescritas=()):
    """Resolve um objeto isolado. prescritas: lista de (faces, T)."""
    basis, A, b = montar(mesh, k, q, conveccoes)
    if not prescritas:
        return basis, solve(A, b)
    x = np.zeros(basis.N)
    D = []
    for faces, T in prescritas:
        dofs = basis.get_dofs(faces).flatten()
        x[dofs] = T
        D.append(dofs)
    return basis, solve(*condense(A, b, x=x, D=np.unique(np.concatenate(D))))


def na_face(mesh, T, faces, funcional, **parametros):
    """Integral de um funcional sobre um conjunto de faces."""
    fb = FacetBasis(mesh, ELEMENTO, facets=faces)
    return asm(funcional, fb, T=fb.interpolate(T), **parametros)


print("Formas definidas: conducao, geracao, conveccao_bilinear, conveccao_linear")

## 5. Verificação contra solução analítica

A verificação confronta o modelo com solução fechada, e não com outra execução do próprio modelo. O caso é o da parede plana de espessura $L$, com a face $x = 0$ a temperatura fixa $T_1$ e a face $x = L$ em convecção com fluido a $T_\infty$. As faces laterais são adiabáticas, de modo que o problema bidimensional reproduz o unidimensional. Pelo circuito térmico de duas resistências em série, o fluxo é

$$q'' = \frac{T_1 - T_\infty}{L/k + 1/h}$$

e a temperatura da face convectiva é $T_L = T_1 - q'' L / k$. O fluxo numérico é a integral de $h\,(T - T_\infty)$ sobre a face convectiva, dividida pela sua altura.

São conferidas três combinações de material e convecção, escolhidas para cobrir os regimes do modelo: alumínio com convecção livre, com número de Biot muito baixo; nicromo com convecção no limite superior da faixa livre; e Teflon com convecção forçada, com número de Biot alto. Um quarto caso acrescenta geração volumétrica, com solução obtida pela integração direta da equação com as mesmas condições de contorno, para conferir também a forma de geração usada na resistência.

Cabe registrar que, em problema unidimensional, o elemento linear reproduz a solução exata nos nós. O erro esperado é, portanto, da ordem do arredondamento, e é também conferido que ele não depende do refinamento da malha. O critério do card é erro relativo abaixo de 1 por cento.

In [ ]:
TOL_VERIFICACAO = 0.01   # 1 por cento


def parede_plana(k, h, T1, T_inf, L, q=0.0, n=40, altura=0.002):
    malha = MeshTri.init_tensor(np.linspace(0.0, L, n + 1), np.linspace(0.0, altura, 3))
    fixa = malha.facets_satisfying(lambda x: np.isclose(x[0], 0.0))
    convectiva = malha.facets_satisfying(lambda x: np.isclose(x[0], L))
    _, T = resolver(malha, k, q, conveccoes=[(convectiva, h, T_inf)], prescritas=[(fixa, T1)])
    fluxo = na_face(malha, T, convectiva, fluxo_convectivo, h=h, T_ref=T_inf) / altura
    T_L = na_face(malha, T, convectiva, integral_T) / altura
    return fluxo, T_L


def analitica_sem_geracao(k, h, T1, T_inf, L):
    fluxo = (T1 - T_inf) / (L / k + 1.0 / h)
    return fluxo, T1 - fluxo * L / k


def analitica_com_geracao(k, h, T1, T_inf, L, q):
    # T(x) = -q x^2 / (2k) + C1 x + T1, com -k T'(L) = h (T(L) - T_inf)
    C1 = (q * L * (1.0 + h * L / (2.0 * k)) - h * (T1 - T_inf)) / (k + h * L)
    T_L = -q * L ** 2 / (2.0 * k) + C1 * L + T1
    return h * (T_L - T_inf), T_L


casos = [
    ("aluminio, conveccao livre", K_AL, H_AR_PARTIDA, 0.0),
    ("nicromo, conveccao livre", K_NICROMO, H_AR_LIVRE[1], 0.0),
    ("Teflon, conveccao forcada", K_TEFLON, H_AR_FORCADA[1], 0.0),
    ("nicromo com geracao", K_NICROMO, H_AR_LIVRE[1], 5.0e5),
]
T1, T_INF, L = 200.0, T_AMBIENTE, 0.01

linhas = []
for nome, k, h, q in casos:
    for n in (10, 40, 160):
        fluxo_num, TL_num = parede_plana(k, h, T1, T_INF, L, q=q, n=n)
        if q == 0.0:
            fluxo_an, TL_an = analitica_sem_geracao(k, h, T1, T_INF, L)
        else:
            fluxo_an, TL_an = analitica_com_geracao(k, h, T1, T_INF, L, q)
        linhas.append({"caso": nome, "elementos": n, "Biot": h * L / k,
                       "q_analitico_W_m2": fluxo_an, "q_numerico_W_m2": fluxo_num,
                       "erro_q_%": 100 * abs(fluxo_num - fluxo_an) / abs(fluxo_an),
                       "T_L_analitico_C": TL_an, "T_L_numerico_C": TL_num,
                       "erro_T_L_%": 100 * abs(TL_num - TL_an) / abs(TL_an)})

verificacao = pd.DataFrame(linhas)
with pd.option_context("display.float_format", "{:.6g}".format):
    print(verificacao.to_string(index=False))

erro_max = verificacao[["erro_q_%", "erro_T_L_%"]].to_numpy().max()
print(f"\nMaior erro relativo: {erro_max:.3e} %  (criterio: abaixo de {100 * TOL_VERIFICACAO:.0f} %)")
assert erro_max < 100 * TOL_VERIFICACAO, "verificacao analitica reprovada"
print("Verificacao analitica APROVADA.")

## 6. Domínio, malha e decisão sobre as placas

### Decisão sobre as placas de alumínio

Na preparação dos dados, a coluna de área recebe preenchimento para a frente, e as quatro linhas Aluminium Plate herdam o rótulo Coil Temps, porque a célula de área está vazia na planilha. O notebook de preparação registra a escolha como intencional. Herdar o rótulo de área, contudo, não implica herdar a física do corpo, e três alternativas foram consideradas: a placa como domínio próprio, como parte do mesmo domínio da resistência, ou fora do modelo.

**A placa é modelada como domínio próprio, acoplado à resistência por contato térmico.** Quatro razões sustentam a escolha.

1. **Critério de aprovação distinto.** O limite da placa é de 150 °C e o da resistência, de 220 °C, uma diferença de 70 °C. As duas temperaturas precisam ser previstas e confrontadas com limites diferentes, o que exige que existam como grandezas separadas no modelo.
2. **Material distinto e variável.** A resistência é um elemento aquecedor metálico; a placa é o suporte, cujo material varia entre alumínio e plástico conforme a coluna `material_coil_support`. As condutividades diferem em cerca de três ordens de grandeza, 237 contra 0,35 W/(m·K). Fundir a placa ao domínio da resistência apagaria justamente a variável de projeto que distingue os ensaios. O nome do termopar, Aluminium Plate, indica a posição da medição, e não o material: a leitura existe também nos ensaios de suporte plástico.
3. **Mecanismo de aquecimento.** A placa não tem fonte própria; ela é aquecida pela resistência. Deixá-la fora do modelo eliminaria o único caminho físico que explica sua temperatura, e as placas respondem por 15 dos 19 falsos aprovados do Coil Temps na linha de base.
4. **Comportamento distinto nos dados.** A Seção 18 do notebook de modelagem mostra que o erro das placas tem natureza diferente do erro das resistências, concentrado em extrapolação para temperaturas acima de 150 °C, e que o material do suporte explica de 19 a 40 por cento da variância das placas.

**Acoplamento.** A face inferior da resistência troca calor com a face de contato da placa por uma condição de Robin cuja referência é a temperatura média da face oposta, com a condutância de contato da Seção 3. O acoplamento entra no sistema como um termo de posto um entre os dois blocos, e o sistema conjunto é resolvido de uma vez, sem iteração. Uma primeira implementação por iteração alternada entre os objetos não convergiu em 200 passos com a condutância de contato adotada, e foi descartada. Por construção, o calor que deixa a resistência pelo contato é igual ao que entra na placa, e o balanço global de energia fecha exatamente, o que é conferido abaixo.

**Geometria.** Cada objeto é representado por um retângulo em corte bidimensional, com profundidade unitária: a resistência sobre a placa, centrada. As dimensões são hipóteses de partida, pois o desenho do produto não foi fornecido pela parceira, e devem ser substituídas quando o forem. As quatro posições, FL, FR, RL e RR, compartilham a geometria nesta entrega; a diferença entre elas fica a cargo da calibração do Card 2.

### Exclusão da radiação

A radiação não integra esta entrega. A troca radiativa é proporcional a $T^4$, o que torna o problema não linear e exigiria iteração sobre a própria solução, fora do orçamento da sprint.

A ausência tem consequência que precisa ficar escrita. Linearizada, a radiação equivale a um coeficiente $h_r = \varepsilon \sigma (T_s + T_{viz})(T_s^2 + T_{viz}^2)$ (BERGMAN et al., 2014, Capítulo 1). Para uma superfície a 220 °C, vizinhança a 25 °C e emissividade unitária, $h_r$ é de cerca de 15 W/(m²·K), da mesma ordem da convecção livre. O coeficiente calibrado no Card 2 será, portanto, um coeficiente efetivo, que absorve parte da troca radiativa, e um valor acima da faixa de convecção livre não indica, por si, erro de calibração. A leitura será registrada nesses termos.

In [ ]:
# Geometria de partida, em metros (hipotese; desenho nao fornecido pela parceira)
GEOMETRIA = {
    "placa_largura": 0.22, "placa_espessura": 0.002,
    "resistencia_largura": 0.18, "resistencia_espessura": 0.008,
}
TAMANHO_ELEMENTO = 0.002   # m, na direcao da largura
TOL = 1e-9


class DoisObjetos:
    """Resistencia sobre placa, em corte bidimensional com profundidade unitaria."""

    def __init__(self, geometria=GEOMETRIA, dx=TAMANHO_ELEMENTO):
        g = geometria
        self.Lp, self.ep = g["placa_largura"], g["placa_espessura"]
        self.Lc, self.ec = g["resistencia_largura"], g["resistencia_espessura"]
        self.x0 = (self.Lp - self.Lc) / 2.0
        nx_p = int(round(self.Lp / dx))
        nx_c = int(round(self.Lc / dx))
        assert np.isclose(self.x0 / dx, round(self.x0 / dx)), "contato precisa coincidir com nos"

        self.malha_placa = MeshTri.init_tensor(
            np.linspace(0.0, self.Lp, nx_p + 1), np.linspace(0.0, self.ep, 3))
        self.malha_resistencia = MeshTri.init_tensor(
            np.linspace(self.x0, self.x0 + self.Lc, nx_c + 1),
            np.linspace(self.ep, self.ep + self.ec, 5))

        mp, mc, x0, Lc, ep, ec, Lp = (self.malha_placa, self.malha_resistencia,
                                      self.x0, self.Lc, self.ep, self.ec, self.Lp)
        no_contato = lambda x: (x[0] >= x0 - TOL) & (x[0] <= x0 + Lc + TOL)
        self.placa_contato = mp.facets_satisfying(
            lambda x: np.isclose(x[1], ep) & no_contato(x))
        topo = mp.facets_satisfying(lambda x: np.isclose(x[1], ep))
        self.placa_livre = np.concatenate([
            np.setdiff1d(topo, self.placa_contato),
            mp.facets_satisfying(lambda x: np.isclose(x[1], 0.0)
                                 | np.isclose(x[0], 0.0) | np.isclose(x[0], Lp))])
        self.resistencia_contato = mc.facets_satisfying(lambda x: np.isclose(x[1], ep))
        self.resistencia_livre = mc.facets_satisfying(
            lambda x: np.isclose(x[1], ep + ec) | np.isclose(x[0], x0) | np.isclose(x[0], x0 + Lc))

        # Conferencias de consistencia da malha e das faces
        comp = lambda m, f: asm(comprimento, FacetBasis(m, ELEMENTO, facets=f)).sum()
        assert np.isclose(comp(mp, self.placa_contato), Lc)
        assert np.isclose(comp(mc, self.resistencia_contato), Lc)
        assert np.isclose(comp(mp, self.placa_livre), (Lp - Lc) + Lp + 2 * ep)
        assert np.isclose(comp(mc, self.resistencia_livre), Lc + 2 * ec)
        assert np.intersect1d(self.placa_contato, self.placa_livre).size == 0

    def resolver(self, potencia, k_placa, k_resistencia=K_RESISTENCIA,
                 h_ar=H_AR_PARTIDA, h_contato=H_CONTATO, T_inf=T_AMBIENTE):
        """Potencia em W por metro de profundidade, dissipada na resistencia."""
        mp, mc, Lc = self.malha_placa, self.malha_resistencia, self.Lc
        q = potencia / (self.Lc * self.ec)
        bc, Ac, fc = montar(mc, k_resistencia, q,
                            [(self.resistencia_livre, h_ar, T_inf),
                             (self.resistencia_contato, h_contato, None)])
        bp, Ap, fp = montar(mp, k_placa, 0.0,
                            [(self.placa_livre, h_ar, T_inf),
                             (self.placa_contato, h_contato, None)])
        gc = asm(comprimento, FacetBasis(mc, ELEMENTO, facets=self.resistencia_contato))
        gp = asm(comprimento, FacetBasis(mp, ELEMENTO, facets=self.placa_contato))
        acopla_cp = sp.csr_matrix(-h_contato / Lc * np.outer(gc, gp))
        acopla_pc = sp.csr_matrix(-h_contato / Lc * np.outer(gp, gc))
        M = sp.bmat([[Ac, acopla_cp], [acopla_pc, Ap]], format="csr")
        T = spsolve(M, np.concatenate([fc, fp]))
        Tc, Tp = T[:bc.N], T[bc.N:]

        T_contato_res, T_contato_placa = gc @ Tc / Lc, gp @ Tp / Lc
        calor_ar_res = na_face(mc, Tc, self.resistencia_livre, fluxo_convectivo, h=h_ar, T_ref=T_inf)
        calor_ar_placa = na_face(mp, Tp, self.placa_livre, fluxo_convectivo, h=h_ar, T_ref=T_inf)
        calor_contato = h_contato * Lc * (T_contato_res - T_contato_placa)
        return {
            "T_resistencia": Tc, "T_placa": Tp,
            "resistencia_max_C": float(Tc.max()),
            "placa_max_C": float(Tp.max()),
            "placa_contato_C": float(T_contato_placa),
            "placa_min_C": float(Tp.min()),
            "potencia_W_m": float(potencia),
            "calor_ar_resistencia_W_m": float(calor_ar_res),
            "calor_ar_placa_W_m": float(calor_ar_placa),
            "calor_contato_W_m": float(calor_contato),
            "balanco_global": float((potencia - calor_ar_res - calor_ar_placa) / potencia),
            "balanco_placa": float((calor_contato - calor_ar_placa) / calor_contato),
        }


modelo = DoisObjetos()
print(f"Malha da resistencia: {modelo.malha_resistencia.p.shape[1]} nos, "
      f"{modelo.malha_resistencia.t.shape[1]} elementos")
print(f"Malha da placa:       {modelo.malha_placa.p.shape[1]} nos, "
      f"{modelo.malha_placa.t.shape[1]} elementos")

### Demonstração e conferências do modelo acoplado

A execução abaixo usa uma potência nominal, escolhida apenas para que as temperaturas fiquem na faixa dos ensaios. **Os valores não são predição**: a potência e o coeficiente de convecção de cada família de configuração são calibrados no Card 2, sobre os ensaios de treino.

Três conferências acompanham a demonstração: o balanço global de energia, em que a potência dissipada precisa igualar o calor entregue ao ar; o balanço da placa, em que o calor recebido por contato precisa igualar o que ela entrega ao ar; e o princípio do máximo, pelo qual a placa, sem fonte própria, não pode ficar mais quente que a face da resistência que a aquece. A última linha repete o caso de alumínio com a malha duas vezes mais fina, para mostrar que o resultado não depende do refinamento.

In [ ]:
POTENCIA_NOMINAL = 690.0   # W/m; apenas demonstrativa, calibrada no Card 2

resultados = []
for rotulo, k_placa, m in [("aluminio", K_PLACA["aluminium"], modelo),
                           ("aluminio 2024-T6", K_AL_2024, modelo),
                           ("plastico (Teflon)", K_PLACA["plastic"], modelo),
                           ("plastico (borracha)", K_BORRACHA, modelo),
                           ("aluminio, malha fina", K_PLACA["aluminium"],
                            DoisObjetos(dx=TAMANHO_ELEMENTO / 2))]:
    r = m.resolver(POTENCIA_NOMINAL, k_placa)
    r["suporte"] = rotulo
    r["T_contato_resistencia_C"] = r["placa_contato_C"] + r["calor_contato_W_m"] / (H_CONTATO * m.Lc)
    resultados.append(r)

demo = pd.DataFrame(resultados)
colunas = ["suporte", "resistencia_max_C", "placa_max_C", "placa_contato_C", "placa_min_C",
           "calor_ar_resistencia_W_m", "calor_ar_placa_W_m", "balanco_global", "balanco_placa"]
print(demo[colunas].round(4).to_string(index=False))

assert (demo["balanco_global"].abs() < 1e-9).all(), "balanco global de energia nao fecha"
assert (demo["balanco_placa"].abs() < 1e-9).all(), "balanco da placa nao fecha"
assert (demo["placa_max_C"] <= demo["T_contato_resistencia_C"] + 1e-9).all(), \
    "principio do maximo violado na placa"
d_malha = abs(demo.loc[4, "placa_max_C"] - demo.loc[0, "placa_max_C"])
print(f"\nDiferenca de temperatura maxima da placa ao refinar a malha: {d_malha:.4f} C")
print("Balanco de energia e principio do maximo conferidos.")
print("\nLeitura qualitativa: com suporte plastico, o calor fica retido junto a resistencia,")
print("que aquece mais, e a placa esfria rapidamente fora da regiao de contato.")

## 7. O que os próximos cards consomem

- `LIMITES_COIL`, `LIMITE` e `LINHA_APROVACAO`: limite ativo e linha de 0,9 vez o limite de cada um dos oito alvos.
- `DoisObjetos.resolver`: temperatura da resistência e da placa para uma potência, um material de suporte, um coeficiente de convecção e uma condutância de contato. É a função calibrada no Card 2, apenas sobre os 46 ensaios de treino, e usada no Card 3 para gerar as features físicas.
- `faixa_h`: classificação de um coeficiente calibrado contra as faixas físicas, para que valor fora delas seja sinalizado.

A verificação analítica da Seção 5 e as conferências da Seção 6 precisam continuar aprovadas depois de qualquer alteração nas formas ou na geometria.

# Card 2 — Calibração do coeficiente de convecção

## 8. Partição e ensaios de treino

O coeficiente de convecção $h$ não é medido na base; ele só passa a existir por calibração, resolvendo o problema inverso: encontrar o $h$ com o qual a temperatura prevista pelo modelo reduzido se aproxima da temperatura medida.

**Restrição.** A calibração usa exclusivamente os 46 ensaios de treino da partição registrada em `colabs_S3/preparacao_testes/particao_treino_teste.json`. A partição é lida do arquivo; nenhum sorteio é feito e nenhuma semente é definida neste notebook, que não usa número aleatório algum nas Seções 1 a 11. Os 10 ensaios de teste não entram em etapa alguma.

**Família de configuração.** A família é a combinação do material do suporte com o tipo de ensaio. As duas variáveis são as que determinam a física do problema: o material define a condutividade da placa, e o tipo de ensaio define quais resistências estão ligadas e com que potência. São também as de maior peso no modelo atual, que concentra nelas 56 por cento da importância. Uma família mais fina, por configuração completa, deixaria 17 grupos para 46 ensaios, a maioria com um ou dois ensaios, sem dado para calibrar. As demais variáveis de projeto, como direção do soprador e abertura do defletor, variam dentro da família, e o efeito delas aparece no resíduo.

**Alinhamento.** Os arquivos de perfil não trazem a coluna `run_idx`, e a posição da linha é o índice do ensaio, como no notebook de modelagem. O alinhamento é conferido contra o tipo de ensaio que a própria partição registra para cada índice.

In [ ]:
import json

with open(resolve("particao_treino_teste.json"), encoding="utf-8") as f:
    PARTICAO = json.load(f)

TREINO = sorted(int(i) for i in PARTICAO["run_idx_treino"])
TESTE = sorted(int(i) for i in PARTICAO["run_idx_teste"])
DOBRA = {int(k): (None if v is None else int(v)) for k, v in PARTICAO["dobra_por_run_idx"].items()}
DOBRAS = sorted({d for d in DOBRA.values() if d is not None})
assert len(TREINO) == 46 and len(TESTE) == 10 and not set(TREINO) & set(TESTE)
assert len(DOBRAS) == 17 and all(DOBRA[i] is None for i in TESTE)

COIL_ORIGINAL = pd.read_csv(resolve("dataset_coil_temps_perfil.csv"))
assert len(COIL_ORIGINAL) == PARTICAO["n_ensaios"]

POSICOES = ["FL", "FR", "RL", "RR"]
COL_RES = {p: f"Coil {p}" for p in POSICOES}
COL_PLACA = {p: f"Aluminium Plate {p}" for p in POSICOES}


def decodifica(df, prefixo):
    colunas = [c for c in df.columns if c.startswith(prefixo)]
    assert (df[colunas].sum(axis=1) == 1).all(), f"{prefixo}: codificacao inconsistente"
    return df[colunas].idxmax(axis=1).str.replace(prefixo, "", regex=False)


coil = COIL_ORIGINAL.copy()
coil.index.name = "run_idx"
coil["material"] = decodifica(coil, "material_coil_support_")
coil["run_case"] = decodifica(coil, "run_case_")
coil["familia"] = coil["material"] + " | " + coil["run_case"]
coil["conjunto"] = np.where(coil.index.isin(TREINO), "treino", "teste")
coil["dobra"] = coil.index.map(DOBRA)
coil["leituras"] = coil[list(COL_RES.values()) + list(COL_PLACA.values())].notna().sum(axis=1)

registrado = pd.Series({int(k): v for k, v in PARTICAO["run_case_por_run_idx"].items()}).sort_index()
assert (coil["run_case"].to_numpy() == registrado.to_numpy()).all(), "linhas desalinhadas da particao"

cobertura = (coil.groupby("familia")
             .agg(treino=("conjunto", lambda s: int((s == "treino").sum())),
                  teste=("conjunto", lambda s: int((s == "teste").sum())))
             .join(coil[coil.conjunto == "treino"].groupby("familia")
                   .agg(treino_completos=("leituras", lambda s: int((s == 8).sum())),
                        configuracoes_treino=("dobra", "nunique")))
             .fillna(0).astype(int).reset_index())
print(cobertura.to_string(index=False))
print(f"\nTotal: {cobertura.treino.sum()} ensaios de treino e {cobertura.teste.sum()} de teste, "
      f"em {len(cobertura)} familias.")

### Asserção de não contaminação

Toda seleção de ensaios usada na calibração passa pela função `garante_somente_treino`, que recusa qualquer índice do conjunto de teste e qualquer índice fora do treino registrado. A calibração da Seção 9 chama essa função antes de ler um único valor medido.

Uma asserção só verifica alguma coisa se falha quando deve. Por isso, cada um dos 10 ensaios de teste é injetado, um de cada vez, junto aos 46 de treino, e a célula exige que todas as injeções sejam recusadas. Um índice inexistente na partição também é injetado. A mesma prova é repetida sobre a própria função de calibração na Seção 9.

In [ ]:
TREINO_SET, TESTE_SET = frozenset(TREINO), frozenset(TESTE)


def garante_somente_treino(indices):
    """Devolve os indices ordenados; falha se algum nao for de treino."""
    usados = {int(i) for i in indices}
    assert usados, "nenhum ensaio informado"
    contaminados = usados & TESTE_SET
    assert not contaminados, f"ensaios de teste na calibracao: {sorted(contaminados)}"
    fora = usados - TREINO_SET
    assert not fora, f"ensaios fora do treino registrado: {sorted(fora)}"
    return sorted(usados)


# 1. O conjunto de treino e aceito
assert garante_somente_treino(TREINO) == TREINO

# 2. Cada ensaio de teste, injetado de proposito, precisa ser recusado
recusados = []
for injetado in TESTE:
    try:
        garante_somente_treino(TREINO + [injetado])
    except AssertionError as erro:
        recusados.append((injetado, str(erro)))
    else:
        raise RuntimeError(f"a assercao aceitou o ensaio de teste {injetado}")
assert len(recusados) == len(TESTE)

# 3. Um indice inexistente na particao tambem e recusado
try:
    garante_somente_treino(TREINO + [999])
except AssertionError:
    pass
else:
    raise RuntimeError("a assercao aceitou um indice inexistente")

print(f"Treino aceito: {len(TREINO)} ensaios.")
print(f"Injecoes recusadas: {len(recusados)} de {len(TESTE)} ensaios de teste, e o indice inexistente.")
print("Exemplo de recusa:", recusados[0][1])

## 9. Calibração por família

### Método

O modelo da Seção 6 é linear na potência: para um $h$ fixo, a elevação de temperatura sobre o ambiente é proporcional à potência dissipada. Basta, portanto, resolver o modelo uma vez por valor de $h$ e por material, com potência unitária, para obter as curvas de resposta $u_r(h)$ da resistência e $u_p(h)$ da placa, em kelvin por watt por metro.

A base não informa a potência das resistências, que muda com a posição e com o tipo de ensaio. Por isso, cada família recebe, além do $h$, uma potência por posição (FL, FR, RL e RR), tratada como parâmetro auxiliar. Para cada $h$ da grade, a potência de cada posição sai por mínimos quadrados em forma fechada, usando juntas as leituras da resistência e da placa daquela posição em todos os ensaios de treino da família:

$$P_j(h) = \frac{u_r \sum_i (T^{r}_{ij} - T_\infty) + u_p \sum_i (T^{p}_{ij} - T_\infty)}{n^r_j\, u_r^2 + n^p_j\, u_p^2}$$

O $h$ fica identificado pela razão entre a elevação da placa e a da resistência, que depende dele; a potência fixa a escala. Com cinco parâmetros por família, um $h$ e quatro potências, e até oito leituras por ensaio, a família fica sobredeterminada a partir de um ensaio completo; a exigência de suficiência abaixo é mais rigorosa que isso.

**Correspondência entre modelo e termopar.** A posição exata dos termopares não foi informada. A leitura da resistência é comparada à temperatura máxima do domínio da resistência, e a da placa à temperatura máxima da placa. É a escolha conservadora: entre os pontos possíveis, é a que prevê a temperatura mais alta e, portanto, a que menos subestima.

**Resíduo.** O resíduo é sempre previsto menos medido, com sinal. Resíduo negativo é subestimação, que é o erro perigoso do risco R06.

**Escolha do $h$ à luz do R06.** Para cada família, calcula-se a raiz do erro quadrático médio em toda a grade. São considerados equivalentes os valores de $h$ cuja raiz fica até 5 por cento acima do mínimo. Dentro do conjunto equivalente, dá-se preferência aos valores que estão na faixa física de 2 a 250 W/(m²·K), quando existe algum; entre eles, é escolhido o $h$ de menor subestimação média, medida pela média dos resíduos negativos, com os positivos contados como zero. Quando nenhum valor equivalente está na faixa física, a escolha é feita sobre o conjunto inteiro e sinalizada. A preferência pela faixa física evita que um valor impossível para o ar seja escolhido apenas por diferença de subestimação que os dados não sustentam. Quando esse valor difere do $h$ de erro mínimo, os dois são registrados na tabela, e a escolha recai sobre o de menor subestimação, pois o R06 pede avaliação assimétrica: entre dois ajustes igualmente bons, o que erra menos para baixo aprova menos arquiteturas que ultrapassam o limite.

### Critérios de suficiência

Uma família só é calibrada quando atende aos dois critérios abaixo. Caso contrário, ela aparece na tabela como não calibrada, com o motivo, e nenhum valor é atribuído a ela.

1. **Dados.** Pelo menos 3 ensaios de treino com as oito leituras presentes. Com um único ensaio, o ajuste de cinco parâmetros sobre oito leituras reproduz aquele ensaio e não permite distinguir o efeito de $h$ da variação entre ensaios.
2. **Aplicabilidade física.** O modelo supõe que a resistência é a fonte de calor e a placa é aquecida por ela. A família só é calibrada se a elevação média da resistência superar a da placa em pelo menos 3 das 4 posições. Quando a placa é mais quente que a resistência, o calor vem de outra fonte, e o modelo não representa o ensaio.

### Sinalizações

- **Faixa física.** O $h$ escolhido é classificado contra as faixas da Tabela 1.1 pela função `faixa_h` da Seção 3. Valor fora de 2 a 250 W/(m²·K) é sinalizado.
- **Identificabilidade.** Se o conjunto equivalente abrange mais de uma década de $h$, os dados não determinam o coeficiente, e a família é sinalizada como de $h$ pouco identificável.
- **Limite da grade.** A grade vai de 1 a 5.000 W/(m²·K). Escolha em uma das extremidades indica que os dados não limitam o $h$ daquele lado.

In [ ]:
GRADE_H = np.geomspace(1.0, 5000.0, 121)   # W/(m2.K)
TOL_EQUIVALENCIA = 0.05                     # raiz do erro ate 5% acima do minimo
MIN_ENSAIOS_COMPLETOS = 3
MIN_POSICOES_FONTE = 3
DECADAS_IDENTIFICAVEL = 1.0


def curvas_de_resposta(grade=GRADE_H, h_contato=H_CONTATO, malha=None):
    """Elevacao de temperatura, em K por W/m, da resistencia e da placa, por material."""
    malha = malha or modelo
    curvas = {}
    for material, k_placa in K_PLACA.items():
        linhas = []
        for h in grade:
            r = malha.resolver(1.0, k_placa, h_ar=h, h_contato=h_contato)
            linhas.append((r["resistencia_max_C"] - T_AMBIENTE, r["placa_max_C"] - T_AMBIENTE))
        curvas[material] = np.array(linhas)
    return curvas


RESPOSTA = curvas_de_resposta()

# Conferencia da linearidade na potencia, premissa do metodo
_r1 = modelo.resolver(1.0, K_AL, h_ar=GRADE_H[40])
_r3 = modelo.resolver(3.0, K_AL, h_ar=GRADE_H[40])
assert np.isclose(_r3["resistencia_max_C"] - T_AMBIENTE, 3 * (_r1["resistencia_max_C"] - T_AMBIENTE))
assert np.isclose(_r3["placa_max_C"] - T_AMBIENTE, 3 * (_r1["placa_max_C"] - T_AMBIENTE))


def potencias(g, u_r, u_p):
    """Potencia por posicao, em W/m, por minimos quadrados em forma fechada."""
    saida = {}
    for p in POSICOES:
        y = g[COL_RES[p]].to_numpy(float) - T_AMBIENTE
        z = g[COL_PLACA[p]].to_numpy(float) - T_AMBIENTE
        my, mz = ~np.isnan(y), ~np.isnan(z)
        saida[p] = (u_r * y[my].sum() + u_p * z[mz].sum()) / (my.sum() * u_r ** 2 + mz.sum() * u_p ** 2)
    return saida


def preve(g, pot, u_r, u_p):
    """Tabela longa com medido, previsto e residuo com sinal de cada leitura."""
    linhas = []
    for run, linha in g.iterrows():
        for p in POSICOES:
            for objeto, coluna, u in (("resistencia", COL_RES[p], u_r), ("placa", COL_PLACA[p], u_p)):
                linhas.append({"run_idx": run, "familia": linha["familia"], "posicao": p,
                               "objeto": objeto, "componente": coluna,
                               "medido": linha[coluna], "previsto": T_AMBIENTE + pot[p] * u})
    t = pd.DataFrame(linhas)
    t = t[t["medido"].notna()].reset_index(drop=True)
    t["residuo"] = t["previsto"] - t["medido"]
    return t


def curva_da_familia(g, resposta):
    """Erro em toda a grade de h. Mesmo calculo de potencias() e preve(), vetorizado."""
    Y = g[[COL_RES[p] for p in POSICOES]].to_numpy(float) - T_AMBIENTE
    Z = g[[COL_PLACA[p] for p in POSICOES]].to_numpy(float) - T_AMBIENTE
    my, mz = ~np.isnan(Y), ~np.isnan(Z)
    soma_y, soma_z = np.nansum(Y, axis=0), np.nansum(Z, axis=0)
    linhas = []
    for i, h in enumerate(GRADE_H):
        u_r, u_p = resposta[g["material"].iloc[0]][i]
        P = (u_r * soma_y + u_p * soma_z) / (my.sum(axis=0) * u_r ** 2 + mz.sum(axis=0) * u_p ** 2)
        r = np.concatenate([(P * u_r - Y)[my], (P * u_p - Z)[mz]])
        linhas.append({"i": i, "h": h, "rmse": float(np.sqrt(np.mean(r ** 2))),
                       "vies": float(r.mean()), "subestimacao": float(np.minimum(r, 0).mean())})
    return pd.DataFrame(linhas)


def suficiencia(g):
    completos = int((g["leituras"] == 8).sum())
    fonte = sum(
        (g[COL_RES[p]] - T_AMBIENTE).mean() > (g[COL_PLACA[p]] - T_AMBIENTE).mean()
        for p in POSICOES)
    if completos < MIN_ENSAIOS_COMPLETOS:
        return False, (f"dados: {completos} ensaio(s) completo(s), "
                       f"minimo {MIN_ENSAIOS_COMPLETOS}")
    if fonte < MIN_POSICOES_FONTE:
        return False, (f"fisica: resistencia mais quente que a placa em {fonte} de 4 posicoes, "
                       f"minimo {MIN_POSICOES_FONTE}; o calor vem de outra fonte")
    return True, ""


def calibrar(indices, resposta=None):
    """Calibra h por familia usando apenas os ensaios indicados, que precisam ser de treino."""
    indices = garante_somente_treino(indices)
    resposta = RESPOSTA if resposta is None else resposta
    dados = coil.loc[indices]
    tabela, parametros = [], {}
    for familia, g in dados.groupby("familia"):
        base = {"familia": familia, "n_treino": len(g),
                "n_completos": int((g["leituras"] == 8).sum())}
        ok, motivo = suficiencia(g)
        if not ok:
            tabela.append({**base, "status": "nao calibrada", "motivo": motivo})
            continue
        curva = curva_da_familia(g, resposta)
        i_min = int(curva["rmse"].idxmin())
        equivalentes = curva[curva["rmse"] <= (1 + TOL_EQUIVALENCIA) * curva["rmse"].min()]
        na_faixa = equivalentes[equivalentes["h"].between(H_AR_LIVRE[0], H_AR_FORCADA[1])]
        candidatos = na_faixa if len(na_faixa) else equivalentes
        i_r06 = int(candidatos["subestimacao"].idxmax())
        h_lo, h_hi = equivalentes["h"].min(), equivalentes["h"].max()
        u_r, u_p = resposta[g["material"].iloc[0]][i_r06]
        pot = potencias(g, u_r, u_p)
        alertas = []
        if faixa_h(GRADE_H[i_r06]) == "FORA DA FAIXA FISICA":
            alertas.append("fora da faixa fisica")
        if np.log10(h_hi / h_lo) > DECADAS_IDENTIFICAVEL:
            alertas.append("h pouco identificavel")
        if i_r06 in (0, len(GRADE_H) - 1):
            alertas.append("no limite da grade")
        tabela.append({
            **base, "status": "calibrada", "motivo": "",
            "h_erro_minimo": curva.loc[i_min, "h"], "rmse_erro_minimo": curva.loc[i_min, "rmse"],
            "subest_erro_minimo": curva.loc[i_min, "subestimacao"],
            "h": curva.loc[i_r06, "h"], "rmse": curva.loc[i_r06, "rmse"],
            "vies": curva.loc[i_r06, "vies"], "subestimacao": curva.loc[i_r06, "subestimacao"],
            "h_equivalente_min": h_lo, "h_equivalente_max": h_hi,
            "equivalentes_na_faixa": len(na_faixa),
            "faixa": faixa_h(GRADE_H[i_r06]), "alertas": "; ".join(alertas),
        })
        parametros[familia] = {"i": i_r06, "h": GRADE_H[i_r06], "u_r": u_r, "u_p": u_p,
                               "potencias": pot, "curva": curva}
    return pd.DataFrame(tabela), parametros


# A funcao de calibracao tambem recusa um ensaio de teste injetado
try:
    calibrar(TREINO + [TESTE[0]])
except AssertionError:
    print("calibrar() recusou o ensaio de teste injetado.")
else:
    raise RuntimeError("calibrar() aceitou um ensaio de teste")

CALIBRACAO, PARAMETROS = calibrar(TREINO)

COLUNAS_TABELA = ["familia", "n_treino", "n_completos", "status", "h", "faixa", "rmse", "vies",
                  "subestimacao", "h_erro_minimo", "rmse_erro_minimo", "subest_erro_minimo",
                  "h_equivalente_min", "h_equivalente_max", "equivalentes_na_faixa", "alertas", "motivo"]
with pd.option_context("display.max_colwidth", 90, "display.width", 250):
    print(CALIBRACAO[COLUNAS_TABELA].round(2).to_string(index=False))

print("\nPotencia calibrada por posicao, em W/m (parametro auxiliar):")
print(pd.DataFrame({f: v["potencias"] for f, v in PARAMETROS.items()}).T.round(1).to_string())

## 10. Resíduo com sinal nos ensaios de treino

O resíduo é reportado com sinal e separado por direção. A média do módulo esconderia justamente o erro perigoso, pois uma subestimação de 20 °C e uma superestimação de 20 °C teriam o mesmo peso. Para cada família e cada componente, são contadas as leituras subestimadas e as superestimadas, com a média de cada grupo, a pior subestimação e as subestimações perigosas: leituras que atingem 0,9 vez o limite enquanto o modelo prevê abaixo dessa linha, que é o caso descrito pelo R06.

Estes números são de ajuste, calculados sobre os mesmos ensaios usados na calibração. A estimativa honesta de erro, fora da amostra, está na Seção 11.

In [ ]:
residuos = pd.concat(
    [preve(coil.loc[TREINO][coil.loc[TREINO, "familia"] == f], par["potencias"], par["u_r"], par["u_p"])
     for f, par in PARAMETROS.items()],
    ignore_index=True)
residuos["linha_aprovacao"] = residuos["componente"].map(LINHA_APROVACAO)
residuos["subestimacao_perigosa"] = ((residuos["medido"] >= residuos["linha_aprovacao"])
                                     & (residuos["previsto"] < residuos["linha_aprovacao"]))


def resumo_com_sinal(t):
    r = t["residuo"]
    neg, pos = r[r < 0], r[r > 0]
    return pd.Series({
        "leituras": len(r), "vies": r.mean(),
        "n_subestima": len(neg), "media_subestimacao": neg.mean() if len(neg) else 0.0,
        "pior_subestimacao": neg.min() if len(neg) else 0.0,
        "n_superestima": len(pos), "media_superestimacao": pos.mean() if len(pos) else 0.0,
        "acima_da_linha": int((t["medido"] >= t["linha_aprovacao"]).sum()),
        "subestimacoes_perigosas": int(t["subestimacao_perigosa"].sum()),
    })


print("Por familia:")
print(residuos.groupby("familia").apply(resumo_com_sinal, include_groups=False).round(2).to_string())
print("\nPor componente:")
print(residuos.groupby("componente").apply(resumo_com_sinal, include_groups=False).round(2).to_string())

sem_previsao = sorted(set(TREINO) - set(residuos["run_idx"]))
print(f"\nEnsaios de treino com previsao: {residuos['run_idx'].nunique()} de {len(TREINO)}.")
print("Sem previsao, por pertencerem a familia nao calibrada:",
      coil.loc[sem_previsao].groupby("familia").size().to_dict())

In [ ]:
import matplotlib.pyplot as plt

# Paleta categorica de referencia, ordem fixa; o marcador e codificacao secundaria
CORES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
MARCADORES = ["o", "s", "^", "D"]
familias = sorted(PARAMETROS)
estilo = {f: (CORES[i], MARCADORES[i]) for i, f in enumerate(familias)}

fig, eixos = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for eixo, objeto, titulo in [(eixos[0], "resistencia", "Resistências (limite 220 °C)"),
                             (eixos[1], "placa", "Placas (limite 150 °C)")]:
    d = residuos[residuos["objeto"] == objeto]
    for f in familias:
        g = d[d["familia"] == f]
        cor, marcador = estilo[f]
        eixo.scatter(g["medido"], g["residuo"], s=36, color=cor, marker=marcador,
                     edgecolor="white", linewidth=0.8, label=f"{f} ({g['run_idx'].nunique()} ensaios)")
    limite = 220.0 if objeto == "resistencia" else 150.0
    eixo.axhline(0.0, color="#1a1a19", linewidth=1.2, label="resíduo zero")
    eixo.axvline(limite, color="#6b6a64", linestyle="--", linewidth=1, label="limite ativo")
    eixo.axvline(0.9 * limite, color="#6b6a64", linestyle=":", linewidth=1, label="0,9 × limite")
    eixo.set_title(titulo, loc="left", fontsize=11)
    eixo.set_xlabel("Temperatura medida (°C)")
    eixo.grid(True, color="#e6e5df", linewidth=0.6)
    eixo.spines[["top", "right"]].set_visible(False)
eixos[0].set_ylabel("Resíduo: previsto − medido (°C)")
eixos[1].legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), frameon=False, fontsize=9)
fig.suptitle(f"Resíduo da calibração nos ensaios de treino: {residuos['run_idx'].nunique()} de "
             f"{len(TREINO)} ensaios com previsão; abaixo de zero, o modelo subestima",
             x=0.01, ha="left", fontsize=12)
fig.text(0.01, -0.02, f"Sem previsão ({len(sem_previsao)} ensaios): famílias não calibradas, "
         "conforme a tabela da Seção 9.", fontsize=9, color="#6b6a64")
fig.tight_layout()
plt.show()

## 11. Resíduo fora da dobra e confronto com o modelo de produção

A geração de dados sintéticos, prevista para depois do Card 3, só se justifica onde o modelo físico é confiável: o resíduo do modelo reduzido contra os ensaios reais precisa ser materialmente menor que o erro atual do modelo de produção naquele componente. O resíduo da Seção 10 não serve a essa comparação, porque é de ajuste, enquanto o erro do Random Forest é medido fora da amostra.

Por isso, a calibração é refeita dentro de cada uma das 17 dobras de validação: a configuração da dobra sai, a calibração usa apenas os demais ensaios de treino, sempre pela função `calibrar`, e o modelo prevê os ensaios que ficaram de fora. Uma família que deixa de atender aos critérios de suficiência em uma dobra não produz previsão naquela dobra.

O Random Forest de produção da área é reproduzido com o protocolo da Seção 19 do notebook de modelagem, e a reprodução é conferida contra os valores registrados. A comparação é feita apenas sobre as leituras que os dois modelos preveem, com a função canônica de avaliação, copiada sem alteração de `colabs_S3/preparacao_testes/metricas.ipynb`.

**Critério adotado para "materialmente menor".** O erro absoluto médio do modelo reduzido precisa ficar abaixo de 80 por cento do erro do Random Forest nas mesmas leituras, isto é, uma redução de pelo menos 20 por cento. O limiar é uma proposta deste card e precisa ser confirmado pelo grupo antes de orientar a geração de dados.

In [ ]:
import os
import json

import numpy as np
import pandas as pd


MARGEM_KPI = 0.10   # margem de erro acordada no TAPI
FATOR_REGRA = 0.90  # regra de projeto: aprovar so 10 por cento abaixo do limite (risco R06)


def _valida(y_real, y_previsto):
    """Converte para vetores e recusa entradas que produziriam metrica sem sentido.

    A conversao para numpy e necessaria porque a subtracao entre duas Series do pandas
    alinha pelos indices, e nao pela posicao. Uma Series com indice original e outra
    com indice reiniciado produziriam diferencas entre pares errados, em silencio.

    O alvo precisa ser estritamente positivo. A restricao e legitima neste projeto,
    pois trata-se de temperatura absoluta em graus Celsius, cujo minimo na base e de
    27,69 graus. Ela existe para transformar em erro visivel o engano de passar valores
    padronizados, caso em que o alvo se aproximaria de zero e o erro percentual
    explodiria em vez de acusar o problema.
    """
    y_real = np.asarray(y_real, dtype=float).ravel()
    y_previsto = np.asarray(y_previsto, dtype=float).ravel()

    if y_real.shape != y_previsto.shape:
        raise ValueError(f"dimensoes diferentes: {y_real.shape} e {y_previsto.shape}")
    if y_real.size == 0:
        raise ValueError("nenhuma observacao a avaliar")
    if np.isnan(y_real).any():
        raise ValueError("ha valores ausentes em y_real; a mascara deve ser aplicada antes")
    if np.isnan(y_previsto).any():
        raise ValueError("ha valores ausentes em y_previsto")
    if (y_real <= 0).any():
        raise ValueError(
            "y_real contem valor nao positivo; o erro percentual exige temperatura "
            "absoluta em graus Celsius, e nao valores padronizados")

    return y_real, y_previsto


def avaliar(y_real, y_previsto, limite=None):
    """Metricas de erro de um componente, com o alvo em graus Celsius.

    Sempre devolvidas:

    n            numero de observacoes que sustentam as metricas
    MAE          erro absoluto medio, na unidade do alvo
    RMSE         raiz do erro quadratico medio, na mesma unidade
    R2           fracao da variancia do alvo explicada pelo modelo
    vies         erro medio com sinal, positivo quando o modelo superestima
    RMSE/MAE     razao entre as duas, indicando concentracao dos erros
    MAPE_%       erro percentual absoluto medio
    WAPE_%       erro absoluto total dividido pela soma dos alvos
    dentro_10_%  proporcao de predicoes dentro da margem de 10 por cento do TAPI

    Devolvidas apenas quando um limite de seguranca e informado:

    limite_C          o proprio limite, para conferencia
    n_reprova_real    ensaios em que o componente de fato atinge o limite
    acerto_decisao_%  concordancia entre a decisao real e a prevista
    falso_aprovado    ensaios reprovados na realidade e aprovados pelo modelo
    falso_reprovado   ensaios aprovados na realidade e reprovados pelo modelo
    linha_regra_C         FATOR_REGRA vezes o limite, linha de aprovacao da regra de projeto
    n_reprova_regra_real  ensaios que atingem a linha da regra na realidade
    falso_aprovado_regra  ensaios que atingem a linha na realidade e ficam abaixo dela
                          na predicao; e a aprovacao falsa descrita no risco R06

    O R2 sai ausente quando o alvo assume um unico valor, situacao em que ele e
    indefinido e em que a implementacao do scikit-learn devolve zero, valor que se
    confunde com um resultado legitimo.
    """
    y_real, y_previsto = _valida(y_real, y_previsto)

    erro = y_previsto - y_real
    mae = float(np.mean(np.abs(erro)))

    # A raiz e calculada explicitamente: o parametro squared do mean_squared_error foi
    # removido do scikit-learn na versao 1.9, e o root_mean_squared_error so existe a
    # partir da 1.4. O calculo direto funciona em qualquer versao.
    rmse = float(np.sqrt(np.mean(erro ** 2)))

    if np.unique(y_real).size < 2:
        r2 = float("nan")
    else:
        ss_res = float(np.sum(erro ** 2))
        ss_tot = float(np.sum((y_real - np.mean(y_real)) ** 2))
        r2 = 1.0 - ss_res / ss_tot

    relativo = np.abs(erro) / y_real

    resultado = {
        "n": int(y_real.size),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "vies": float(np.mean(erro)),
        "RMSE/MAE": (rmse / mae) if mae > 0 else float("nan"),
        "MAPE_%": 100.0 * float(np.mean(relativo)),
        "WAPE_%": 100.0 * float(np.sum(np.abs(erro)) / np.sum(y_real)),
        "dentro_10_%": 100.0 * float(np.mean(relativo <= MARGEM_KPI)),
    }

    if limite is not None and np.isfinite(limite):
        reprova_real = y_real >= limite
        reprova_prevista = y_previsto >= limite
        resultado.update({
            "limite_C": float(limite),
            "n_reprova_real": int(reprova_real.sum()),
            "acerto_decisao_%": 100.0 * float(np.mean(reprova_real == reprova_prevista)),
            "falso_aprovado": int(np.sum(reprova_real & ~reprova_prevista)),
            "falso_reprovado": int(np.sum(~reprova_real & reprova_prevista)),
        })

        # Regra de projeto: a mesma linha, 0,9 vez o limite, vale para o real e o previsto
        linha = FATOR_REGRA * float(limite)
        reprova_regra_real = y_real >= linha
        reprova_regra_prevista = y_previsto >= linha
        resultado.update({
            "linha_regra_C": linha,
            "n_reprova_regra_real": int(reprova_regra_real.sum()),
            "falso_aprovado_regra": int(np.sum(reprova_regra_real & ~reprova_regra_prevista)),
        })

    return resultado


def consolidar(tabela, grupo="dataset",
               metricas=("MAE", "RMSE", "R2", "vies", "MAPE_%", "WAPE_%", "dentro_10_%")):
    """Consolida metricas por componente em metricas por area.

    A macro-media atribui o mesmo peso a cada componente e responde se o modelo funciona
    bem em todas as partes do produto. A micro-media pondera pelo numero de observacoes
    e responde qual o erro tipico de uma leitura qualquer.

    O R2 nao e comparavel entre areas, pois seu denominador e a variancia do alvo de
    cada area. Ele indica se o modelo supera a media dentro da propria area.
    """
    if grupo not in tabela.columns:
        raise ValueError(f"coluna de agrupamento ausente: {grupo}")
    if "n" not in tabela.columns:
        raise ValueError("a tabela precisa da coluna n para a micro-media")

    metricas = [m for m in metricas if m in tabela.columns]
    linhas = []

    for nome, bloco in tabela.groupby(grupo):
        linha = {grupo: nome, "componentes": len(bloco), "n_total": int(bloco["n"].sum())}
        for m in metricas:
            valores = bloco[m]
            validos = valores.notna()
            linha[f"{m}_macro"] = (float(valores[validos].mean())
                                   if validos.any() else float("nan"))
            if validos.any() and bloco.loc[validos, "n"].sum() > 0:
                linha[f"{m}_micro"] = float(
                    np.average(valores[validos], weights=bloco.loc[validos, "n"]))
            else:
                linha[f"{m}_micro"] = float("nan")

        # As metricas de decisao sao contagens, e por isso somadas em vez de promediadas
        for contagem in ("n_reprova_real", "falso_aprovado", "falso_reprovado",
                     "n_reprova_regra_real", "falso_aprovado_regra"):
            if contagem in bloco.columns:
                linha[contagem] = int(bloco[contagem].fillna(0).sum())
        linhas.append(linha)

    return pd.DataFrame(linhas).sort_values(grupo).reset_index(drop=True)


print("Metricas definidas: avaliar, consolidar")

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

# Reproducao do Random Forest de producao, restrita ao Coil Temps
# (Secao 19 do notebook de modelagem; mesma logica de colabs_S3/linha_base_kpi.ipynb)
PREFIXOS_FEATURES = ["blower_direction", "housing", "board_position", "run_case", "losses",
                     "perfil_perdas", "deflector_hole", "material_coil_support", "cfg"]
AREA_PARAMS = dict(n_estimators=300, random_state=42, n_jobs=-1)   # hiperparametros de producao
MIN_TREINO, MIN_PREDICOES = 8, 5


def entradas_de(df):
    return [c for c in df.columns if any(c.startswith(p + "_") for p in PREFIXOS_FEATURES)]


def formato_longo(df, extras=()):
    """Uma linha por par de ensaio e componente medido, apenas ensaios de treino."""
    entradas = entradas_de(df)
    blocos = []
    for col in [c for c in df.columns if c in ALVOS]:
        y = df[col]
        sub = df.loc[y.notna(), entradas].copy()
        sub["_run"] = np.arange(len(df))[y.notna()]
        sub["_componente"] = col
        sub["_alvo"] = y[y.notna()].to_numpy()
        blocos.append(sub)
    longo = pd.concat(blocos, ignore_index=True)
    longo["_dobra"] = longo["_run"].map(DOBRA)
    return longo[longo["_dobra"].notna()].reset_index(drop=True)


def atributos_do_componente(longo, treino):
    g = longo[treino].groupby("_componente")["_alvo"]
    return g.mean(), g.std().fillna(1.0).replace(0, 1.0), float(longo.loc[treino, "_alvo"].mean())


def matriz(longo, mascara, entradas, media, desvio, geral):
    comp = longo.loc[mascara, "_componente"]
    return np.hstack([longo.loc[mascara, entradas].to_numpy(float),
                      comp.map(media).fillna(geral).to_numpy().reshape(-1, 1),
                      comp.map(desvio).fillna(1.0).to_numpy().reshape(-1, 1)])


def random_forest_fora_da_dobra(longo, entradas):
    blocos = []
    for d in DOBRAS:
        treino = (longo["_dobra"] != d).to_numpy()
        val = (longo["_dobra"] == d).to_numpy()
        if treino.sum() < MIN_TREINO or not val.any():
            continue
        media, desvio, geral = atributos_do_componente(longo, treino)
        escala = StandardScaler()
        rf = RandomForestRegressor(**AREA_PARAMS)
        rf.fit(escala.fit_transform(matriz(longo, treino, entradas, media, desvio, geral)),
               longo.loc[treino, "_alvo"])
        bloco = longo.loc[val, ["_run", "_componente", "_alvo"]].copy()
        bloco["previsto"] = rf.predict(escala.transform(matriz(longo, val, entradas, media, desvio, geral)))
        blocos.append(bloco)
    out = pd.concat(blocos, ignore_index=True)
    return out.rename(columns={"_run": "run_idx", "_componente": "componente", "_alvo": "medido"})


LONGO = formato_longo(COIL_ORIGINAL)
ENTRADAS = entradas_de(COIL_ORIGINAL)
rf_oof = random_forest_fora_da_dobra(LONGO, ENTRADAS)
_ref = avaliar(rf_oof["medido"], rf_oof["previsto"])
print(f"Random Forest reproduzido: {_ref['n']} predicoes, MAE {_ref['MAE']:.2f} C, "
      f"dentro de 10% {_ref['dentro_10_%']:.2f}%")
assert _ref["n"] == 344 and round(_ref["MAE"], 2) == 11.71 and round(_ref["dentro_10_%"], 2) == 65.41, \
    "reproducao diverge da Secao 19 do notebook de modelagem"

# Modelo reduzido fora da dobra: calibracao refeita sem a configuracao da dobra
fem_oof = []
for d in DOBRAS:
    fora = [i for i in TREINO if DOBRA[i] == d]
    _, par = calibrar([i for i in TREINO if DOBRA[i] != d])
    for f, g in coil.loc[fora].groupby("familia"):
        if f in par:
            fem_oof.append(preve(g, par[f]["potencias"], par[f]["u_r"], par[f]["u_p"]).assign(dobra=d))
fem_oof = pd.concat(fem_oof, ignore_index=True)
assert set(fem_oof["run_idx"]) <= TREINO_SET
print(f"Modelo reduzido fora da dobra: {len(fem_oof)} predicoes em {fem_oof['run_idx'].nunique()} ensaios.")

In [ ]:
FATOR_MATERIAL = 0.80   # criterio proposto: reducao de pelo menos 20% no erro absoluto medio

pares = fem_oof.merge(rf_oof, on=["run_idx", "componente"], suffixes=("_fem", "_rf"))
assert np.allclose(pares["medido_fem"], pares["medido_rf"])
pares = pares.rename(columns={"medido_fem": "medido"})

linhas = []
for comp, g in pares.groupby("componente"):
    lim = LIMITE[comp]
    fem = avaliar(g["medido"], g["previsto_fem"], limite=lim)
    rf = avaliar(g["medido"], g["previsto_rf"], limite=lim)
    linhas.append({"componente": comp, "n": fem["n"],
                   "MAE_fem": fem["MAE"], "MAE_rf": rf["MAE"],
                   "vies_fem": fem["vies"], "vies_rf": rf["vies"],
                   "dentro_10_fem": fem["dentro_10_%"], "dentro_10_rf": rf["dentro_10_%"],
                   "falso_aprovado_fem": fem["falso_aprovado"], "falso_aprovado_rf": rf["falso_aprovado"],
                   "razao_MAE": fem["MAE"] / rf["MAE"]})
PORTAO = pd.DataFrame(linhas)
PORTAO["portao"] = np.where(PORTAO["razao_MAE"] <= FATOR_MATERIAL, "aberto", "fechado")
print("Leituras comparaveis por familia:", pares.groupby("familia").size().to_dict())
print()
print(PORTAO.round(2).to_string(index=False))
todos_fem = avaliar(pares["medido"], pares["previsto_fem"])
todos_rf = avaliar(pares["medido"], pares["previsto_rf"])
print(f"\nNas {len(pares)} leituras comuns: MAE {todos_fem['MAE']:.2f} C no modelo reduzido e "
      f"{todos_rf['MAE']:.2f} C no Random Forest; vies {todos_fem['vies']:+.2f} e {todos_rf['vies']:+.2f} C.")
print(f"Componentes com portao aberto: {int((PORTAO.portao == 'aberto').sum())} de {len(PORTAO)}.")

## 12. Sensibilidade do h calibrado à condutância de contato

A condutância de contato entre resistência e placa é fixada na Seção 3 e não é calibrada neste card. Como o $h$ calibrado depende dela, a calibração é repetida com a condutância reduzida a um décimo, um centésimo e um milésimo do valor de partida, para verificar se um $h$ fora da faixa física é propriedade do ar ou consequência da hipótese de contato. É uma análise de sensibilidade: nenhum dos valores reduzidos é adotado.

In [ ]:
GRADE_SENS = np.geomspace(1.0, 5000.0, 61)
_grade_original = GRADE_H
sens = []
try:
    GRADE_H = GRADE_SENS
    for fator in (1.0, 0.1, 0.01, 0.001):
        resposta = curvas_de_resposta(GRADE_SENS, h_contato=fator * H_CONTATO)
        tab, _ = calibrar(TREINO, resposta=resposta)
        for _, r in tab[tab["status"] == "calibrada"].iterrows():
            sens.append({"fator_contato": fator, "h_contato": fator * H_CONTATO,
                         "familia": r["familia"], "h": r["h"], "faixa": r["faixa"],
                         "rmse": r["rmse"], "vies": r["vies"], "alertas": r["alertas"]})
finally:
    GRADE_H = _grade_original

SENSIBILIDADE = pd.DataFrame(sens)
print(SENSIBILIDADE.round(2).to_string(index=False))

## 13. Registro da calibração

### Resultado por família

| Família | Ensaios de treino | Situação | h escolhido, W/(m²·K) | Faixa | Raiz do erro | Viés | Subestimação média |
|---|---:|---|---:|---|---:|---:|---:|
| alumínio, broil c2 | 13 | calibrada | 735,7 | **fora da faixa física** | 12,88 °C | +0,30 °C | −5,00 °C |
| alumínio, broil qualitative | 17 | calibrada | 910,3 | **fora da faixa física** | 10,32 °C | +0,25 °C | −3,83 °C |
| plástico, broil c2 | 4 | calibrada, h pouco identificável | 2,03 | convecção livre | 10,12 °C | 0,00 °C | −2,43 °C |
| plástico, broil qualitative | 3 | calibrada | 165,7 | convecção forçada | 1,24 °C | 0,00 °C | −0,49 °C |
| alumínio, broil c1 | 1 | **não calibrada** | — | — | — | — | — |
| plástico, broil c1 | 1 | **não calibrada** | — | — | — | — | — |
| alumínio, pyro | 7 | **não calibrada** | — | — | — | — | — |

As duas famílias de broil c1 têm um único ensaio de treino cada e não atendem ao critério de dados. A família pyro tem dados, mas não atende ao critério físico: a placa é mais quente que a resistência em três das quatro posições, e as resistências ficam em torno de 50 °C, o que indica que o calor vem do forno e não do elemento que o modelo representa. Nenhum valor foi atribuído a essas três famílias, e 9 dos 46 ensaios de treino ficam sem previsão.

### A escolha do h à luz do R06

Em todas as famílias calibradas, o $h$ de erro mínimo e o $h$ escolhido são diferentes, e a escolha recaiu sobre o de menor subestimação dentro do conjunto equivalente.

- **Alumínio, broil c2.** O erro mínimo ocorre em 847,9, com subestimação média de −5,11 °C; o escolhido, 735,7, tem −5,00 °C, ao custo de 0,22 °C na raiz do erro. Foi escolhido por subestimar menos.
- **Alumínio, broil qualitative.** Erro mínimo em 1.049,1, com −3,90 °C; escolhido 910,3, com −3,83 °C, ao custo de 0,16 °C. Mesma razão.
- **Plástico, broil c2.** O conjunto equivalente vai de 1,0 a 685,3, quase três décadas: os dados não determinam o $h$. O de menor subestimação no conjunto inteiro seria 1,0, fora da faixa física e na borda da grade; por isso, a escolha foi restrita aos 68 valores equivalentes que estão na faixa física, e recaiu sobre 2,03, com −2,43 °C, contra −2,55 °C do erro mínimo, em 49,6. A diferença de subestimação é pequena, e o valor deve ser lido como não identificado, e não como medida do ar.
- **Plástico, broil qualitative.** Erro mínimo em 124,8 e escolhido 165,7, com a mesma subestimação média de −0,49 °C até a segunda casa; a escolha é indiferente para o R06.

### Confronto com a faixa física

Os dois valores das famílias de alumínio estão acima da faixa de convecção forçada, e não são propriedade do ar. A análise da Seção 12 mostra a causa. Com a condutância de contato de partida, placa e resistência ficam praticamente na mesma temperatura, e o único modo de reproduzir a placa medida, cerca de 30 por cento menos aquecida que a resistência, é aumentar muito a perda para o ar. Reduzindo a condutância de contato a um centésimo, 36 W/(m²·K), o $h$ das duas famílias cai para 8,4 e 11,2 W/(m²·K), dentro da convecção livre, com raiz do erro de 13,25 e 10,33 °C, contra 12,88 e 10,32 °C com o contato de partida. Uma condutância dessa ordem corresponde a uma camada de ar de cerca de 0,7 mm entre a resistência e a placa, tomando a condutividade do ar a 300 K, 0,0263 W/(m·K) (BERGMAN et al., 2014, Tabela A.4), o que sugere que a resistência não está em contato pleno com o suporte.

A conclusão é que **os dados identificam a combinação do coeficiente de convecção com a condutância de contato, e não cada um separadamente.** O $h$ calibrado só tem leitura física condicionada ao valor de contato adotado. A escolha desse valor é uma decisão de modelagem que precisa ser tomada pelo grupo, preferencialmente com informação de montagem da parceira, e não é feita neste card.

### Resíduo e o que ele revela

O resíduo de ajuste tem viés próximo de zero por construção, mas a dispersão é grande: nas famílias de alumínio, metade das leituras é subestimada, com média de −9,6 °C e −7,4 °C e pior caso de −32 °C, numa placa. Nenhuma leitura de ajuste configura subestimação perigosa, isto é, medida acima de 0,9 vez o limite e prevista abaixo, mas esse resultado é de ajuste e não se sustenta fora da amostra.

As faixas diagonais do gráfico têm explicação direta: dentro de uma família, a previsão de cada posição é uma só, e o resíduo cai um grau para cada grau a mais na medição. O modelo reduzido, na forma calibrada, produz um valor por família e posição, e a variação entre ensaios da mesma família, causada pelas demais variáveis de projeto, fica toda no resíduo. É a limitação que o Card 3 precisa considerar: como feature, a temperatura calibrada é função do material, do tipo de ensaio e da posição, variáveis que o modelo de produção já recebe.

### Portão para a geração de dados sintéticos

Fora da dobra, nas 224 leituras em que os dois modelos preveem, o modelo reduzido tem erro absoluto médio de 10,21 °C, contra 9,06 °C do Random Forest. O critério proposto, redução de pelo menos 20 por cento, é atendido apenas na Aluminium Plate RL, com razão de 0,72. **O portão fica fechado em 7 dos 8 componentes, e a geração de dados sintéticos a partir deste modelo não se justifica.**

Há, porém, um resultado que o erro médio esconde. Nas mesmas leituras, o modelo reduzido produz 2 falsos aprovados, contra 13 do Random Forest, todos os 11 de diferença nas placas FL e RL. Nessas duas placas, o viés do modelo reduzido é positivo, +6,36 e +3,30 °C, e o do Random Forest é negativo, −3,77 e −6,61 °C. Pelo critério do R06, erro médio maior acompanhado de menos aprovação falsa não é piora automática, e essa troca é o que o Card 3 precisa medir ao combinar os dois modelos.

As famílias de plástico têm uma única configuração de treino cada. Quando essa configuração é deixada de fora, não sobra ensaio para calibrar, e o desempenho delas fora da amostra não pode ser medido com a partição atual.

### Limitações

- A calibração e a avaliação usam ensaios simulados, e medem consistência entre simulações, e não realismo.
- A correspondência entre termopar e ponto do modelo é uma hipótese conservadora, o máximo de cada domínio.
- A temperatura ambiente de 25 °C e a geometria são hipóteses de partida, não informadas pela parceira.
- O limiar de 20 por cento do portão é uma proposta e precisa ser confirmado pelo grupo.

# Card 3 — Features físicas no modelo de produção

## 14. Função canônica com a regra de projeto

A função canônica de `colabs_S3/preparacao_testes/metricas.ipynb` passou a devolver também as métricas de decisão sobre a linha da regra de projeto, 0,9 vez o limite: `n_reprova_regra_real` e `falso_aprovado_regra`. A mudança foi feita na fonte, testada por caso próprio na bateria daquele notebook e propagada sem alteração às cópias, inclusive à da Seção 11 deste notebook, de modo que as três grandezas deste card são medidas com uma única definição.

As três grandezas são:

1. **Margem contratada**: `dentro_10_%`, proporção de predições dentro de 10 por cento do valor medido.
2. **Subestimação**: `vies`, com sinal, por componente.
3. **Aprovação falsa**: `falso_aprovado`, contra o limite absoluto, e `falso_aprovado_regra`, contra 0,9 vez o limite.

A célula abaixo confere que a cópia em uso é a versão atual.

In [ ]:
_teste = avaliar([100.0, 120.0], [98.0, 109.0], limite=110.0)
assert "falso_aprovado_regra" in _teste, "a copia da funcao canonica esta desatualizada"
assert _teste["falso_aprovado_regra"] == 1 and np.isclose(_teste["linha_regra_C"], 99.0)
print({k: _teste[k] for k in ("limite_C", "falso_aprovado", "linha_regra_C",
                              "n_reprova_regra_real", "falso_aprovado_regra")})

## 15. Features físicas fora da dobra

Cada linha do formato longo, um par de ensaio e componente, recebe três features do modelo reduzido:

- `fem_temperatura`: a temperatura prevista pelo modelo calibrado para aquele componente, em °C;
- `fem_log_h`: o logaritmo decimal do coeficiente de convecção calibrado da família;
- `fem_calibrado`: 1 quando a família do ensaio foi calibrada, 0 caso contrário.

**Sem vazamento.** Dentro de cada dobra da validação, a calibração é refeita com a função `calibrar` apenas sobre os ensaios de treino daquela dobra, e as features dos ensaios deixados de fora vêm dessa calibração. Os ensaios de teste não entram em momento algum.

**Validação cruzada interna.** A feature física é construída a partir do próprio alvo, como uma codificação pelo alvo. Se as linhas de treino de uma dobra recebessem a temperatura de uma calibração que usou os seus próprios valores medidos, a feature pareceria mais precisa no treino do que é fora dele, e o Random Forest aprenderia a confiar nela além da conta. Por isso, dentro de cada dobra, a feature de cada linha de treino vem de uma calibração que exclui também a configuração dessa linha: são 17 calibrações externas e 272 internas, todas pela função `calibrar`. A versão sem essa proteção foi medida e é reportada como diagnóstico na Seção 16.

**Famílias não calibradas.** Onde a família não foi calibrada, a temperatura e o coeficiente ficam ausentes, e nenhum valor é imputado. O Random Forest do scikit-learn aceita valor ausente nativamente, e o indicador `fem_calibrado` informa a condição ao modelo.

**O que a feature carrega.** Como registrado na Seção 13, a temperatura calibrada é função do material, do tipo de ensaio e da posição do componente. A feature não traz informação nova de projeto: ela reorganiza informação que o modelo já recebe em uma forma com estrutura física, a razão entre a elevação da placa e a da resistência e a escala da potência por posição. A pergunta deste card é se essa reorganização muda o comportamento do modelo, e não se ela acrescenta variável.

In [ ]:
COLUNAS_FEM = ["fem_temperatura", "fem_log_h", "fem_calibrado"]
PARAMETROS_DOBRA = {d: calibrar([i for i in TREINO if DOBRA[i] != d])[1] for d in DOBRAS}
PARAMETROS_INTERNOS = {(d, c): calibrar([i for i in TREINO if DOBRA[i] not in (d, c)])[1]
                       for d in DOBRAS for c in DOBRAS if c != d}


def features_fisicas(longo, parametros):
    """Features do modelo reduzido para cada linha do formato longo."""
    familia = longo["_run"].map(coil["familia"])
    temp = np.full(len(longo), np.nan)
    log_h = np.full(len(longo), np.nan)
    calibrado = np.zeros(len(longo))
    for k, (f, comp) in enumerate(zip(familia, longo["_componente"])):
        par = parametros.get(f)
        if par is None:
            continue
        posicao = comp.split()[-1]
        u = par["u_r"] if ALVOS[comp] == "resistencia" else par["u_p"]
        temp[k] = T_AMBIENTE + par["potencias"][posicao] * u
        log_h[k] = np.log10(par["h"])
        calibrado[k] = 1.0
    return pd.DataFrame({"fem_temperatura": temp, "fem_log_h": log_h, "fem_calibrado": calibrado},
                        index=longo.index)


def features_da_dobra(d, cruzado=True):
    """Features de todas as linhas para a dobra d.

    Linhas da dobra: calibracao sem a dobra. Linhas de treino: com cruzado, calibracao
    sem a dobra e sem a configuracao da propria linha; sem cruzado, a mesma da dobra.
    """
    partes = []
    for c in DOBRAS:
        linhas = LONGO[LONGO["_dobra"] == c]
        par = PARAMETROS_DOBRA[d] if (c == d or not cruzado) else PARAMETROS_INTERNOS[(d, c)]
        partes.append(features_fisicas(linhas, par))
    return pd.concat(partes).loc[LONGO.index]


def features_de_treino_completo():
    """Features para o modelo treinado nos 46 ensaios: cada linha sem a propria configuracao."""
    partes = [features_fisicas(LONGO[LONGO["_dobra"] == c], PARAMETROS_DOBRA[c]) for c in DOBRAS]
    return pd.concat(partes).loc[LONGO.index]


# Conferencias: a calibracao que gera a feature de uma linha nunca usou o ensaio dela
assert set(LONGO["_run"]) <= TREINO_SET, "o formato longo contem ensaio fora do treino"
for (d, c) in PARAMETROS_INTERNOS:
    usados = {i for i in TREINO if DOBRA[i] not in (d, c)}
    assert not usados & {i for i in TREINO if DOBRA[i] in (d, c)}
print(f"Calibracoes: {len(PARAMETROS_DOBRA)} externas e {len(PARAMETROS_INTERNOS)} internas.")

cobertura_fem = []
for d in DOBRAS:
    val = LONGO["_dobra"] == d
    feat = features_fisicas(LONGO[val], PARAMETROS_DOBRA[d])
    cobertura_fem.append(feat.assign(familia=LONGO.loc[val, "_run"].map(coil["familia"]).to_numpy()))
cobertura_fem = pd.concat(cobertura_fem)
print("Linhas de validacao com feature fisica, por familia:")
print(cobertura_fem.groupby("familia")["fem_calibrado"].agg(linhas="size", com_feature="sum")
      .astype(int).to_string())
print(f"\nTotal: {int(cobertura_fem['fem_calibrado'].sum())} de {len(cobertura_fem)} linhas de validacao.")

## 16. Comparação com e sem o modelo físico

### Três configurações

- **A. Sem FEM**: o modelo de produção, sem alteração.
- **B. FEM somado aos rótulos**: as entradas de produção mais as três features físicas.
- **C. FEM no lugar dos rótulos**: as entradas de produção sem as colunas de tipo de ensaio (`run_case_`) e de perdas (`losses_`), mais as três features físicas.

As três usam o mesmo algoritmo, os mesmos hiperparâmetros, a mesma semente de produção, as mesmas 17 dobras e as mesmas 344 leituras de validação. A configuração A precisa reproduzir exatamente o resultado da Seção 11, o que é conferido.

Uma quarta linha, **B sem validação cruzada interna**, repete a configuração B com as features de treino tiradas da mesma calibração da dobra, sem excluir a configuração da própria linha. Ela não concorre na decisão: serve para medir o efeito da proteção descrita na Seção 15.

### Critério de decisão

A comparação é feita contra a configuração A, na área inteira e no recorte das famílias calibradas:

- **piora**: aumenta o `falso_aprovado` ou o `falso_aprovado_regra`, qualquer que seja o efeito no erro. Queda de erro médio acompanhada de aumento de aprovação falsa é piora, e não melhora;
- **melhora**: nenhuma das duas aprovações falsas aumenta e a proporção dentro da margem aumenta;
- **sem ganho**: nenhum dos casos anteriores.

Com 344 leituras, uma diferença de uma ou duas aprovações falsas, ou de um ponto percentual na margem, está dentro da variação esperada entre execuções de configurações próximas, e é lida como tal.

In [ ]:
REMOVIDAS_C = ("run_case_", "losses_")
CONFIGURACOES = {
    "A. sem FEM": {"fem": False, "remover": (), "cruzado": True},
    "B. FEM somado aos rotulos": {"fem": True, "remover": (), "cruzado": True},
    "C. FEM no lugar dos rotulos": {"fem": True, "remover": REMOVIDAS_C, "cruzado": True},
    "B sem validacao cruzada interna": {"fem": True, "remover": (), "cruzado": False},
}
PRINCIPAIS = [n for n in CONFIGURACOES if n[:2] in ("A.", "B.", "C.")]


def colunas_da_configuracao(cfg):
    return [c for c in ENTRADAS if not (cfg["remover"] and c.startswith(cfg["remover"]))]


def matriz_cfg(longo, mascara, entradas, fem, media, desvio, geral):
    blocos = [longo.loc[mascara, entradas].to_numpy(float)]
    if fem is not None:
        blocos.append(fem.loc[mascara, COLUNAS_FEM].to_numpy(float))
    comp = longo.loc[mascara, "_componente"]
    blocos += [comp.map(media).fillna(geral).to_numpy().reshape(-1, 1),
               comp.map(desvio).fillna(1.0).to_numpy().reshape(-1, 1)]
    return np.hstack(blocos)


def validar_configuracao(cfg):
    entradas = colunas_da_configuracao(cfg)
    blocos = []
    for d in DOBRAS:
        treino = (LONGO["_dobra"] != d).to_numpy()
        val = (LONGO["_dobra"] == d).to_numpy()
        if treino.sum() < MIN_TREINO or not val.any():
            continue
        fem = features_da_dobra(d, cfg["cruzado"]) if cfg["fem"] else None
        media, desvio, geral = atributos_do_componente(LONGO, treino)
        escala = StandardScaler()
        rf = RandomForestRegressor(**AREA_PARAMS)
        rf.fit(escala.fit_transform(matriz_cfg(LONGO, treino, entradas, fem, media, desvio, geral)),
               LONGO.loc[treino, "_alvo"])
        bloco = LONGO.loc[val, ["_run", "_componente", "_alvo"]].copy()
        bloco["previsto"] = rf.predict(
            escala.transform(matriz_cfg(LONGO, val, entradas, fem, media, desvio, geral)))
        blocos.append(bloco)
    return (pd.concat(blocos, ignore_index=True)
            .rename(columns={"_run": "run_idx", "_componente": "componente", "_alvo": "medido"}))


PREVISOES = {nome: validar_configuracao(cfg) for nome, cfg in CONFIGURACOES.items()}

# A configuracao A reproduz exatamente o modelo de producao da Secao 11
_a = PREVISOES["A. sem FEM"].merge(rf_oof, on=["run_idx", "componente"], suffixes=("", "_ref"))
assert len(_a) == 344 and np.allclose(_a["previsto"], _a["previsto_ref"]), "configuracao A diverge"
for nome, p in PREVISOES.items():
    assert len(p) == 344 and set(p["run_idx"]) <= TREINO_SET
print("Configuracoes validadas:", ", ".join(PREVISOES), "| 344 leituras cada.")

In [ ]:
CALIBRADAS = set(CALIBRACAO.loc[CALIBRACAO["status"] == "calibrada", "familia"])


def avaliar_recorte(p, recorte):
    p = p.assign(familia=p["run_idx"].map(coil["familia"]))
    if recorte == "familias calibradas":
        p = p[p["familia"].isin(CALIBRADAS)]
    por_comp = pd.DataFrame([
        {"componente": c, "objeto": ALVOS[c], **avaliar(g["medido"], g["previsto"], limite=LIMITE[c])}
        for c, g in p.groupby("componente")])
    total = avaliar(p["medido"], p["previsto"])
    return por_comp, {
        "leituras": len(p), "dentro_10_%": total["dentro_10_%"], "MAE": total["MAE"],
        "vies": total["vies"], "n_reprova_real": int(por_comp["n_reprova_real"].sum()),
        "falso_aprovado": int(por_comp["falso_aprovado"].sum()),
        "n_reprova_regra_real": int(por_comp["n_reprova_regra_real"].sum()),
        "falso_aprovado_regra": int(por_comp["falso_aprovado_regra"].sum()),
    }


def veredito(linha, base):
    if (linha["falso_aprovado"] > base["falso_aprovado"]
            or linha["falso_aprovado_regra"] > base["falso_aprovado_regra"]):
        return "piora"
    if linha["dentro_10_%"] > base["dentro_10_%"]:
        return "melhora"
    return "sem ganho"


RESUMO, POR_COMPONENTE = [], {}
for recorte in ("area inteira", "familias calibradas"):
    linhas = {}
    for nome, p in PREVISOES.items():
        por_comp, total = avaliar_recorte(p, recorte)
        POR_COMPONENTE[(recorte, nome)] = por_comp
        linhas[nome] = total
    base = linhas["A. sem FEM"]
    for nome, total in linhas.items():
        RESUMO.append({"recorte": recorte, "configuracao": nome, **total,
                       "veredito": ("referencia" if nome.startswith("A.") else
                                    veredito(total, base) if nome in PRINCIPAIS else "diagnostico")})
RESUMO = pd.DataFrame(RESUMO)
print(RESUMO.round(2).to_string(index=False))

In [ ]:
# As tres grandezas por componente, area inteira
tabelas = []
for nome in PRINCIPAIS:
    t = POR_COMPONENTE[("area inteira", nome)].set_index("componente")
    tabelas.append(t[["dentro_10_%", "vies", "falso_aprovado", "falso_aprovado_regra"]]
                   .add_prefix(nome[:1] + "_"))
POR_COMP_TABELA = pd.concat(tabelas, axis=1)
POR_COMP_TABELA.insert(0, "n_reprova_regra_real",
                       POR_COMPONENTE[("area inteira", "A. sem FEM")]
                       .set_index("componente")["n_reprova_regra_real"])
print("Prefixos A, B e C conforme a configuracao; vies em graus Celsius, com sinal.")
print(POR_COMP_TABELA.round(2).to_string())

## 17. Importância das variáveis e verificação de mecanismo

O diagnóstico do modelo atual é de sobreajuste, e não de falta de informação: o modelo se apoia nos rótulos de tipo de ensaio e de perdas para montar uma tabela de consulta por configuração. Medir só a queda de erro, portanto, não decide nada. A verificação de mecanismo pergunta se a soma da importância de `run_case` e `losses` cai quando as features físicas entram.

O valor de 86 por cento citado no diagnóstico do projeto vem do Gradient Boosting com um modelo por componente, na Seção 12 de `testes_treinamento/treino_gradient_boosting.ipynb`. O modelo de produção é outro, e tem entradas adicionais, o patamar e a escala do componente. A referência deste card é, por isso, a configuração A medida aqui, com o mesmo cálculo das outras duas.

A importância é a redução média de impureza do Random Forest, somada por grupo de variável, em modelos treinados sobre os 46 ensaios de treino. Nas configurações com FEM, a feature de cada linha vem da calibração que exclui a configuração da própria linha, pelo mesmo motivo da Seção 15. Ela é reportada de duas formas: sobre todas as entradas, e apenas sobre as variáveis de projeto e físicas, sem o patamar e a escala, que identificam o componente e dominam a soma.

In [ ]:
def grupo_da_variavel(nome):
    for prefixo in PREFIXOS_FEATURES:
        if nome.startswith(prefixo + "_"):
            return prefixo
    return nome


def importancia_da_configuracao(cfg):
    entradas = colunas_da_configuracao(cfg)
    tudo = np.ones(len(LONGO), dtype=bool)
    fem = features_de_treino_completo() if cfg["fem"] else None
    media, desvio, geral = atributos_do_componente(LONGO, tudo)
    rf = RandomForestRegressor(**AREA_PARAMS)
    rf.fit(StandardScaler().fit_transform(matriz_cfg(LONGO, tudo, entradas, fem, media, desvio, geral)),
           LONGO["_alvo"])
    nomes = entradas + (COLUNAS_FEM if cfg["fem"] else []) + ["patamar", "escala"]
    grupos = ["fem" if n.startswith("fem_") else grupo_da_variavel(n) for n in nomes]
    return pd.Series(rf.feature_importances_, index=grupos).groupby(level=0).sum()


IMPORTANCIA = pd.DataFrame({nome: importancia_da_configuracao(CONFIGURACOES[nome])
                            for nome in PRINCIPAIS}).fillna(0.0)
sobre_todas = 100 * IMPORTANCIA / IMPORTANCIA.sum()
projeto = IMPORTANCIA.drop(index=["patamar", "escala"])
sobre_projeto = 100 * projeto / projeto.sum()

print("Importancia sobre todas as entradas (%):")
print(sobre_todas.sort_values("A. sem FEM", ascending=False).round(1).to_string())
print("\nImportancia sobre as variaveis de projeto e fisicas (%):")
print(sobre_projeto.sort_values("A. sem FEM", ascending=False).round(1).to_string())

MECANISMO = pd.DataFrame({
    "run_case + losses, sobre todas (%)": sobre_todas.reindex(["run_case", "losses"]).sum(),
    "run_case + losses, sobre projeto (%)": sobre_projeto.reindex(["run_case", "losses"]).sum(),
    "fem, sobre projeto (%)": sobre_projeto.reindex(["fem"]).sum(),
    "run_case + losses + fem, sobre projeto (%)":
        sobre_projeto.reindex(["run_case", "losses", "fem"]).sum(),
})
print("\nVerificacao de mecanismo:")
print(MECANISMO.round(1).to_string())

## 18. Registro do Card 3

### As três grandezas, em validação cruzada sobre as 344 leituras

| Configuração | Dentro de 10% | MAE | Viés | Falso aprovado | Falso aprovado, regra 0,9 | Veredito |
|---|---:|---:|---:|---:|---:|---|
| A. sem FEM | 65,41% | 11,71 °C | +0,53 °C | 19 | 22 | referência |
| B. FEM somado aos rótulos | 60,47% | 15,23 °C | −2,57 °C | 24 | 27 | **piora** |
| C. FEM no lugar dos rótulos | 54,36% | 25,43 °C | +4,06 °C | 23 | 27 | **piora** |
| B sem validação cruzada interna | 67,44% | 14,07 °C | −2,23 °C | 22 | 32 | diagnóstico |

No recorte das 280 leituras das famílias calibradas, o resultado é o mesmo: a margem cai de 73,57 para 63,93 e 65,36 por cento, e a aprovação falsa contra a regra sobe de 19 para 26 e 24.

**As features físicas não melhoram o modelo de produção, e pioram as três grandezas ao mesmo tempo.** Não há ganho a defender: a margem contratada diminui, o viés fica negativo nas configurações com FEM somado aos rótulos, isto é, o modelo passa a subestimar, e a aprovação falsa aumenta nas duas leituras, absoluta e contra a regra de projeto.

A linha de diagnóstico mostra por que o critério do card é assimétrico. Sem a validação cruzada interna, a configuração B exibe a melhor margem da tabela, 67,44 por cento, acima da referência, e ao mesmo tempo a pior aprovação falsa contra a regra, 32 contra 22. É exatamente o caso que o critério recusa: margem melhor com decisão de segurança pior não é melhora. A diferença entre as duas versões de B mede o efeito de construir a feature a partir do alvo sem proteção: a feature parece mais informativa do que é, e o modelo a segue.

### Verificação de mecanismo

O diagnóstico do projeto é de sobreajuste: o modelo se apoia em dois rótulos categóricos sobre 17 configurações de treino. A pergunta era se as features físicas reduzem essa dependência.

| Configuração | run_case + losses, sobre as variáveis de projeto | fem | Soma |
|---|---:|---:|---:|
| A. sem FEM | 87,4% | 0,0% | 87,4% |
| B. FEM somado aos rótulos | 58,9% | 36,1% | 95,0% |
| C. FEM no lugar dos rótulos | 0,0% | 67,0% | 67,0% |

A soma da importância de `run_case` e `losses` cai, e na configuração C vai a zero, mas **a queda não indica que o modelo passou a descrever física.** A temperatura calibrada é função do material do suporte, do tipo de ensaio e da posição do componente, conforme registrado na Seção 13: ela é outra forma de escrever os mesmos rótulos. Na configuração B, a soma dos três grupos sobe para 95,0 por cento, acima dos 87,4 por cento da referência, ou seja, a concentração aumenta em lugar de diminuir. Na configuração C, o modelo simplesmente troca o rótulo pela feature derivada dele, e o erro dobra.

A conclusão é que o valor de 87,4 por cento da configuração A não mede um defeito que a feature física, na forma calibrada deste card, consiga corrigir. O que falta ao modelo não é uma reescrita física dos rótulos existentes, e sim variável de projeto que a base não traz.

### O que isso significa para o Nível 3

O objetivo do Nível 3 era usar o modelo reduzido como motor de features físicas e de dados sintéticos. Com o que foi medido nos Cards 1 a 3, **nenhuma das duas frentes se sustenta hoje**:

- como feature, o modelo reduzido piora as três grandezas;
- como gerador de dados sintéticos, o portão da Seção 11 está fechado em 7 dos 8 componentes, porque o resíduo fora da dobra do modelo reduzido é maior que o erro do modelo de produção.

Três razões, todas registradas nos cards anteriores, explicam o resultado e indicam o que precisaria mudar antes de uma nova tentativa.

1. **O coeficiente de convecção não é identificável junto da condutância de contato** (Seção 12). Enquanto a hipótese de contato não for fixada com informação de montagem da parceira, o modelo tem um grau de liberdade a mais que os dados não fecham.
2. **A calibração só distingue material e tipo de ensaio.** As variáveis de vazão, direção do soprador e abertura do defletor não entram no modelo físico, e é justamente nelas que estaria a informação nova. Fazer o coeficiente depender delas exige mais ensaios por família, e é o pedido a levar à parceira.
3. **A radiação está fora do modelo** (Seção 6), e a 220 °C ela tem a mesma ordem de grandeza da convecção livre.

Registra-se um resultado favorável, medido no Card 2 e não usado aqui: o modelo reduzido, sozinho, produz menos aprovação falsa que o Random Forest nas mesmas leituras, 2 contra 13, por subestimar menos junto ao limite. O caminho que isso sugere não é usá-lo como feature, e sim como segunda opinião na decisão de aprovação, hipótese que não foi medida neste card e que precisa de card próprio.

### Limitações

- O experimento cobre a área Coil Temps. Nada aqui autoriza conclusão sobre as outras seis áreas.
- A importância por redução de impureza é enviesada a favor de variáveis contínuas, e a feature física é contínua enquanto os rótulos são binários. A leitura foi feita em termos de direção, e não de valor exato.
- As diferenças de uma ou duas aprovações falsas estão no limite do que 344 leituras distinguem; as diferenças aqui registradas como piora são maiores que isso.
- A avaliação é de validação cruzada. O conjunto de teste reservado não foi tocado neste card.